In [1]:
# Célula 1: Montar o Google Drive
print("Montando Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive montado com sucesso. Verifique a pasta 'drive/MyDrive'.")

Montando Google Drive...
Mounted at /content/drive
Google Drive montado com sucesso. Verifique a pasta 'drive/MyDrive'.


In [2]:
# Célula 2: Instalar bibliotecas necessárias
# Esta célula instala todas as bibliotecas Python que utilizaremos no projeto.
# 'pandas': Para manipulação de dados em formato tabular (CSV).
# 'neo4j': Driver oficial para interagir com o banco de dados Neo4j.
# 'rdflib': Para trabalhar com dados RDF/OWL (nossa ontologia .ttl).
# 'transformers': Biblioteca do Hugging Face para carregar e usar modelos Transformer.
# 'langchain_community': Componentes da LangChain (embeddings, LLMs).
# 'langchain_google_genai': Integração da LangChain com modelos Gemini.
# 'networkx': Uma biblioteca para criação e manipulação de grafos em Python (útil para representação intermediária).
# 'tabulate': Para exibir dados em formato de tabela no console.
# 'ipywidgets': Para elementos interativos no Colab.
# 'unidecode': Para normalização de texto (remover acentos).
# 'numpy': Para operações numéricas, especialmente com embeddings.
# 'google-generativeai': Acesso direto à API do Gemini para listar modelos.

print("Instalando bibliotecas...")
# O '!' executa comandos de shell (terminal) no Colab
!pip install -q pandas neo4j rdflib transformers langchain_community langchain_google_genai networkx tabulate ipywidgets unidecode numpy google-generativeai
print("Bibliotecas instaladas.")

Instalando bibliotecas...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
Bibliotecas instaladas.


In [3]:
# Célula 3: Configurar Credenciais do Neo4j usando Colab Secrets
# Esta célula carrega suas credenciais do Neo4j de forma segura,
# utilizando o recurso 'Secrets' do Google Colab.
# É VITAL que suas chaves (NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD) estejam configuradas nos Secrets.

from google.colab import userdata
import os

print("Carregando credenciais Neo4j dos Secrets do Colab...")

# Tenta carregar as credenciais
try:
    NEO4J_URI = userdata.get('NEO4J_URI')
    NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
    NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

    # Verifica se todas as credenciais foram carregadas
    if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
        raise ValueError("Neo4j credentials not fully set in Colab Secrets.")

    print("Credenciais Neo4j carregadas com sucesso dos Secrets do Colab. ✅")
    print("URI: [URI OMITIDA POR SEGURANÇA]") # Não imprima o URI completo
    print(f"Usuário: {NEO4J_USERNAME}")

except Exception as e:
    print(f"ERRO ao carregar credenciais Neo4j: {e}")
    print("POR FAVOR, configure NEO4J_URI, NEO4J_USERNAME e NEO4J_PASSWORD nos Secrets do Colab.")
    print("Clique no ícone de chave 🔑 no painel esquerdo do Colab (abaixo do ícone de pasta).")
    print("Para continuar, você DEVE configurar esses secrets. O código de conexão falhará sem eles.")
    # Define valores vazios para evitar erro de NameError em células futuras, mas a conexão falhará
    NEO4J_URI = ""
    NEO4J_USERNAME = ""
    NEO4J_PASSWORD = ""

Carregando credenciais Neo4j dos Secrets do Colab...
Credenciais Neo4j carregadas com sucesso dos Secrets do Colab. ✅
URI: [URI OMITIDA POR SEGURANÇA]
Usuário: neo4j


In [4]:
# # Célula 3.1 Instala a bibliotecas
print("Instalando langchain-community...")
!pip install langchain-community
print("langchain-community instalado.")

print("Instalando langchain-google-genai...")
!pip install langchain-google-genai
print("langchain-google-genai instalado.")

print("Instalando unidecode...")
!pip install unidecode
print("unidecode instalado.")


Instalando langchain-community...
langchain-community instalado.
Instalando langchain-google-genai...
langchain-google-genai instalado.
Instalando unidecode...
unidecode instalado.


In [5]:
# Célula 3.2: Listar Modelos Gemini Disponíveis (SÓ EXECUTAR SE PRECISAR VER A LISTA DE MODELOS)
# Esta célula utiliza sua GOOGLE_API_KEY para listar todos os modelos Gemini acessíveis
# que suportam a geração de conteúdo. Útil para depuração futura ou para verificar novas opções.
# **Importante**: A GOOGLE_API_KEY deve ter sido carregada na Célula 3 antes de executar esta.

import google.generativeai as genai
import os # Necessário para acessar as variáveis de ambiente

# --- Configuração da API Key do Google ---
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    if not GOOGLE_API_KEY:
        raise ValueError("GOOGLE_API_KEY não encontrada nos Secrets do Colab.")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("GOOGLE_API_KEY carregada com sucesso dos Secrets do Colab. ✅")
except Exception as e:
    print(f"ERRO ao carregar GOOGLE_API_KEY: {e}")
    print("Por favor, configure 'GOOGLE_API_KEY' nos Secrets do Colab (ícone de chave 🔑).")
    raise ValueError("Chave de API do Google ausente.")


print("Listando modelos Gemini disponíveis para sua GOOGLE_API_KEY...")
available_models = []

# Verifica se a GOOGLE_API_KEY está definida como variável de ambiente (setada na Célula 3)
if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    print("ERRO: GOOGLE_API_KEY não está definida como variável de ambiente. Verifique a Célula 3.")
else:
    try:
        # Configura a API GenerativeAI com a chave carregada
        genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

        for m in genai.list_models():
            # Filtra por modelos que suportam a função 'generateContent' (essencial para LLMs)
            if "generateContent" in m.supported_generation_methods:
                available_models.append(m.name)

        if available_models:
            print("\nModelos disponíveis que suportam 'generateContent':")
            for model_name in available_models:
                print(f"- {model_name}")
        else:
            print("\nNenhum modelo Gemini que suporte 'generateContent' encontrado para sua GOOGLE_API_KEY. Verifique sua conta ou região.")

    except Exception as e:
        print(f"ERRO ao listar modelos disponíveis: {e}")
        print("Isso pode indicar um problema com sua GOOGLE_API_KEY ou acesso geral à API Generative Language.")

GOOGLE_API_KEY carregada com sucesso dos Secrets do Colab. ✅
Listando modelos Gemini disponíveis para sua GOOGLE_API_KEY...

Modelos disponíveis que suportam 'generateContent':
- models/gemini-1.5-pro-latest
- models/gemini-1.5-pro-002
- models/gemini-1.5-pro
- models/gemini-1.5-flash-latest
- models/gemini-1.5-flash
- models/gemini-1.5-flash-002
- models/gemini-1.5-flash-8b
- models/gemini-1.5-flash-8b-001
- models/gemini-1.5-flash-8b-latest
- models/gemini-2.5-pro-preview-03-25
- models/gemini-2.5-flash-preview-05-20
- models/gemini-2.5-flash
- models/gemini-2.5-flash-lite-preview-06-17
- models/gemini-2.5-pro-preview-05-06
- models/gemini-2.5-pro-preview-06-05
- models/gemini-2.5-pro
- models/gemini-2.0-flash-exp
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-exp-image-generation
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.0-flash-preview-image-generation
- models/gemini-2.0-flash-lite-preview-02-05
- models/

In [6]:
# Célula 4: Carregar Modelos de Linguagem - Gemini 1.5 Flash Latest e Embeddings
# Esta célula configura o LLM para usar o modelo Gemini 1.5 Flash mais recente (via API)
# e o modelo de embeddings para busca semântica.

print("Carregando modelos de linguagem (Gemini 1.5 Flash Latest e Embeddings)...")

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from transformers import pipeline # Removido AutoTokenizer/AutoModelForSeq2SeqLM, pois não são usados diretamente aqui
import torch # Necessário para garantir que embeddings rodem em CPU

# --- Definindo o LLM de Geração (Gemini 1.5 Flash Latest) ---
# Modelo escolhido da lista que você me forneceu.
LLM_MODEL_NAME_TO_USE = "gemini-1.5-flash-latest"

llm = None
try:
    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL_NAME_TO_USE,
        temperature=0.7,
        client_options={"api_endpoint": "generativelanguage.googleapis.com"}
    )
    print(f"LLM de geração '{LLM_MODEL_NAME_TO_USE}' (via API) carregado para LangChain. ✅")

except Exception as e:
    print(f"ERRO ao carregar o LLM '{LLM_MODEL_NAME_TO_USE}': {e}")
    print(f"Verifique sua GOOGLE_API_KEY e se o modelo '{LLM_MODEL_NAME_TO_USE}' está realmente disponível para sua conta.")
    llm = None
    raise e

# --- Carregamento do modelo de Embeddings (paraphrase-MiniLM-L6-v2) ---
embedding_model_name = "sentence-transformers/paraphrase-MiniLM-L6-v2"
embeddings = None
try:
    embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={'device': 'cpu'})
    print(f"Modelo de Embeddings '{embedding_model_name}' carregado com sucesso na CPU. ✅")
    print("Seu 'motor de PLN' para embeddings está pronto.")

except Exception as e:
    print(f"ERRO ao carregar o modelo de Embeddings: {e}")
    print("Verifique o nome do modelo ou sua conexão.")
    embeddings = None
    raise e

# --- Testes de funcionalidade ---
if llm:
    print(f"\nTestando o LLM {LLM_MODEL_NAME_TO_USE} com uma consulta de exemplo:")
    try:
        test_prompt = "Explique brevemente o conceito de Grafos de Conhecimento."
        result = llm.invoke(test_prompt)
        print(f"Resposta do {LLM_MODEL_NAME_TO_USE} para '{test_prompt}':\n{result.content}")
    except Exception as e:
        print(f"Erro ao executar teste LLM {LLM_MODEL_NAME_TO_USE}: {e}")

if embeddings:
    try:
        test_embedding_phrases = ["mochila voadora", "produto caro", "detalhes de cliente"]
        print(f"\nTestando o modelo de Embeddings com algumas frases de exemplo:")
        for phrase in test_embedding_phrases:
            vec = embeddings.embed_query(phrase)
            print(f"- Frase: '{phrase}' -> Vetor (primeiros 5 elementos): {vec[:5]}...")
            print(f"  (Tamanho do vetor: {len(vec)})")
    except Exception as e:
        print(f"Erro ao executar teste básico do modelo de Embeddings: {e}")

Carregando modelos de linguagem (Gemini 1.5 Flash Latest e Embeddings)...
LLM de geração 'gemini-1.5-flash-latest' (via API) carregado para LangChain. ✅


/tmp/ipython-input-2214511171.py:35: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={'device': 'cpu'})


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de Embeddings 'sentence-transformers/paraphrase-MiniLM-L6-v2' carregado com sucesso na CPU. ✅
Seu 'motor de PLN' para embeddings está pronto.

Testando o LLM gemini-1.5-flash-latest com uma consulta de exemplo:
Resposta do gemini-1.5-flash-latest para 'Explique brevemente o conceito de Grafos de Conhecimento.':
Grafos de conhecimento são estruturas de dados que representam informações como um grafo, onde os nós (vértices) representam entidades (pessoas, lugares, coisas, conceitos) e as arestas representam as relações entre essas entidades.  Em essência, são grandes bases de dados semiestruturadas que organizam informações de forma a capturar relações complexas entre os dados, permitindo inferências e raciocínio.  A riqueza de um grafo de conhecimento reside na quantidade e qualidade das relações representadas.

Testando o modelo de Embeddings com algumas frases de exemplo:
- Frase: 'mochila voadora' -> Vetor (primeiros 5 elementos): [-0.6531563997268677, 0.08085806667804718, -0.

In [7]:
# Célula 5: Estabelecer a Conexão com o Neo4j
# Esta célula utiliza o driver Python do Neo4j para criar e testar a conexão com a sua instância do banco de dados de grafo.
# Ela depende das credenciais carregadas na Célula 3 (NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD).

print("Estabelecendo conexão com o Neo4j...")
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable, AuthError

# Verifica se as credenciais foram carregadas corretamente na Célula 3.
# Se alguma delas estiver vazia, significa que houve um problema na Célula 3.
if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    print("ERRO CRÍTICO: Credenciais Neo4j não estão disponíveis. Revise a Célula 3 e seus Secrets do Colab.")
    driver = None # Garante que a variável 'driver' seja None para evitar erros em células futuras.
else:
    driver = None # Inicializa 'driver' como None.
    try:
        # Cria uma instância do driver do Neo4j. Este driver é a ponte entre seu código Python e o banco Neo4j.
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

        # Teste de Conexão: Uma consulta Cypher simples para verificar se o banco responde.
        # O 'with driver.session() as session:' garante que a sessão com o banco seja aberta e fechada corretamente.
        with driver.session() as session:
            # A consulta 'RETURN 1 AS test' é o teste mais básico para confirmar que o banco está vivo.
            session.run("RETURN 1 AS test")
            print("Conexão com o Neo4j estabelecida com sucesso! ✅")
            print("Driver Neo4j pronto para uso.")

    except ServiceUnavailable as e:
        print(f"ERRO DE CONEXÃO COM NEO4J: O banco de dados pode não estar disponível (desligado, em manutenção) ou o firewall está bloqueando. Detalhes: {e}")
        print("Certifique-se de que sua instância Neo4j AuraDB está ATIVA no site do AuraDB.")
        driver = None
    except AuthError as e:
        print(f"ERRO DE AUTENTICAÇÃO COM NEO4J: As credenciais (usuário/senha) podem estar incorretas. Detalhes: {e}")
        print("Verifique suas credenciais Neo4j nos Secrets do Colab e no site do AuraDB.")
        driver = None
    except Exception as e: # Captura outros erros inesperados
        print(f"ERRO INESPERADO ao conectar ao Neo4j: Detalhes: {e}")
        driver = None

Estabelecendo conexão com o Neo4j...
Conexão com o Neo4j estabelecida com sucesso! ✅
Driver Neo4j pronto para uso.


In [8]:
# Célula 6: Carregar e Processar Dados do Modelo Lógico
# Esta célula lê o arquivo CSV do modelo lógico expandido do seu Google Drive
# e o carrega em um DataFrame pandas para fácil manipulação.
# Certifique-se de que o arquivo 'modelo_logico_expandido.csv' esteja na pasta do seu projeto no Google Drive.

import pandas as pd
import os

# Definindo o caminho base para a pasta do seu projeto no Google Drive
# CERTIFIQUE-SE DE QUE ESTE CAMINHO ESTÁ CORRETO para onde você colocou seus arquivos!
project_folder_path = '/content/drive/MyDrive/00_Projeto_ICMC' # <--- AJUSTE AQUI SE SUA PASTA TIVER OUTRO NOME

file_path_modelo_logico = os.path.join(project_folder_path, 'plataforma_dados_modelo_logico.csv')

print(f"Carregando o arquivo do modelo lógico de: {file_path_modelo_logico}")

modelo_logico_df = None # Inicializa o DataFrame como None

try:
    # Lê o arquivo CSV para um DataFrame do pandas
    modelo_logico_df = pd.read_csv(file_path_modelo_logico)

    print("Modelo lógico carregado com sucesso! ✅")
    print("\nVisualizando as primeiras 5 linhas do DataFrame:")
    print(modelo_logico_df.head().to_markdown(index=False)) # to_markdown para melhor visualização no Colab

    print(f"\nTotal de linhas no modelo lógico: {len(modelo_logico_df)}")

except FileNotFoundError:
    print(f"ERRO: O arquivo '{file_path_modelo_logico}' NÃO foi encontrado.")
    print("Certifique-se de que o nome do arquivo está correto e que ele está na pasta especificada no seu Google Drive.")
    print("Verifique também se o Google Drive foi montado corretamente (Célula 1).")
except Exception as e:
    print(f"ERRO ao carregar o modelo lógico: {e}")

Carregando o arquivo do modelo lógico de: /content/drive/MyDrive/00_Projeto_ICMC/plataforma_dados_modelo_logico.csv
Modelo lógico carregado com sucesso! ✅

Visualizando as primeiras 5 linhas do DataFrame:
| Nome_Tabela      | Nome_Coluna   | Tipo_Dado    | Chave   |   Referencia_Tabela_FK |   Referencia_Coluna_FK | Descricao_Coluna                      | Dominio_Negocio   |
|:-----------------|:--------------|:-------------|:--------|-----------------------:|-----------------------:|:--------------------------------------|:------------------|
| Cliente          | ID_Cliente    | INT          | PK      |                    nan |                    nan | Identificador único do cliente        | Cliente           |
| Cliente          | Nome_Cliente  | VARCHAR(255) | Null    |                    nan |                    nan | Nome completo do cliente              | Cliente           |
| Cliente          | Email_Cliente | VARCHAR(255) | NK      |                    nan |                    n

In [9]:
# Célula 7: Carregar e Processar Metadados de Governança
# Esta célula lê o arquivo CSV contendo os metadados de governança (DMBOK simulados)
# do seu Google Drive e os carrega em um DataFrame pandas.
# Certifique-se de que o arquivo 'metadados_governanca.csv' esteja na pasta do seu projeto.

import pandas as pd
import os

# Definindo o caminho base para a pasta do seu projeto no Google Drive
# CERTIFIQUE-SE DE QUE ESTE CAMINHO ESTÁ CORRETO! (Deve ser o mesmo da Célula 6)
project_folder_path = '/content/drive/MyDrive/00_Projeto_ICMC' # <--- CONFIRME/AJUSTE AQUI
file_name_metadados_governanca = 'plataforma_dados_metadados_catalogo.csv'
file_path_metadados_governanca = os.path.join(project_folder_path, file_name_metadados_governanca)

print(f"Carregando o arquivo de metadados de governança de: {file_path_metadados_governanca}")

metadados_governanca_df = None # Inicializa o DataFrame como None

try:
    # Lê o arquivo CSV para um DataFrame do pandas
    metadados_governanca_df  = pd.read_csv(file_path_metadados_governanca)

    print("Metadados de governança carregados com sucesso! ✅")
    print("\nVisualizando as primeiras 5 linhas do DataFrame:")
    # .fillna('') preenche valores NaN (Not a Number) com strings vazias para melhor visualização
    print(metadados_governanca_df.head().fillna('').to_markdown(index=False))

    print(f"\nTotal de linhas nos metadados de governança: {len(metadados_governanca_df)}")
    print(f"Colunas detectadas: {metadados_governanca_df.columns.tolist()}")

except FileNotFoundError:
    print(f"ERRO: O arquivo '{file_path_metadados_governanca}' NÃO foi encontrado.")
    print("Certifique-se de que o nome do arquivo está correto e que ele está na pasta especificada.")
    print("Verifique também se o Google Drive foi montado corretamente (Célula 1).")
except Exception as e:
    print(f"ERRO inesperado ao carregar os metadados de governança: {e}")
    print("Verifique manualmente o CSV para problemas de formato.")

Carregando o arquivo de metadados de governança de: /content/drive/MyDrive/00_Projeto_ICMC/plataforma_dados_metadados_catalogo.csv
Metadados de governança carregados com sucesso! ✅

Visualizando as primeiras 5 linhas do DataFrame:
| Nome_Tabela      | Nome_Coluna   | Dominio_Negocio   | Classificacao_Confidencialidade   | Proprietario_Dado   | Regras_Qualidade_Dados                        | Freq_Atualizacao   |   Retencao_Dados_Anos | Regra_Negocio_Associada                                                                                | Termo_Glossario_DMBOK       |
|:-----------------|:--------------|:------------------|:----------------------------------|:--------------------|:----------------------------------------------|:-------------------|----------------------:|:-------------------------------------------------------------------------------------------------------|:----------------------------|
| Cliente          | ID_Cliente    | Cliente           | Restrito                  

In [10]:
# Célula 8: Carregar e Processar a Ontologia
# Esta célula lê o arquivo TTL da ontologia do seu Google Drive e o carrega usando a biblioteca RDFLib.
# A ontologia define os conceitos (classes), propriedades e relações semânticas do seu domínio.
# Certifique-se de que o arquivo 'modelo_logico_ontologia_empresa.ttl' esteja na pasta do seu projeto.

from rdflib import Graph
import os

# Definindo o caminho base para a pasta do seu projeto no Google Drive
# CERTIFIQUE-SE DE QUE ESTE CAMINHO ESTÁ CORRETO! (Deve ser o mesmo das Células 6 e 7)
project_folder_path = '/content/drive/MyDrive/00_Projeto_ICMC' # <--- CONFIRME/AJUSTE AQUI
file_name_ontologia = 'modelo_logico_ontologia_empresa.ttl'
file_path_ontologia = os.path.join(project_folder_path, file_name_ontologia)

print(f"Carregando o arquivo da ontologia de: {file_path_ontologia}")

ontology_graph = Graph() # Cria um grafo RDFLib vazio

try:
    # Carrega a ontologia do arquivo TTL
    # 'format="turtle"' especifica o formato do arquivo para o RDFLib
    ontology_graph.parse(file_path_ontologia, format="turtle")

    print("Ontologia carregada com sucesso! ✅")
    print(f"\nTotal de triplas (sentenças) na ontologia: {len(ontology_graph)}")

    print("\nVisualizando algumas triplas da ontologia (exemplo):")
    # Imprime as primeiras 10 triplas (sujeito, predicado, objeto) para verificar
    count = 0
    for s, p, o in ontology_graph:
        print(f"  {s} -- {p} --> {o}")
        count += 1
        if count >= 10:
            break
    if count == 0:
        print("Nenhuma tripla encontrada no grafo da ontologia. Verifique o conteúdo do arquivo TTL.")

except FileNotFoundError:
    print(f"ERRO: O arquivo '{file_path_ontologia}' NÃO foi encontrado.")
    print("Certifique-se de que o nome do arquivo está correto e que ele está na pasta especificada.")
    print("Verifique também se o Google Drive foi montado corretamente (Célula 1).")
except Exception as e:
    print(f"ERRO inesperado ao carregar a ontologia: {e}")
    print("Verifique o conteúdo do arquivo TTL para erros de sintaxe.")

Carregando o arquivo da ontologia de: /content/drive/MyDrive/00_Projeto_ICMC/modelo_logico_ontologia_empresa.ttl
Ontologia carregada com sucesso! ✅

Total de triplas (sentenças) na ontologia: 787

Visualizando algumas triplas da ontologia (exemplo):
  http://www.exemplo.com/ontologia/mochilas_voadoras#contemItem -- http://www.w3.org/2000/01/rdf-schema#label --> contém item
  http://www.exemplo.com/ontologia/mochilas_voadoras#Veiculo -- http://www.w3.org/2000/01/rdf-schema#comment --> Subclasse de Produto representando veículos.
  http://www.exemplo.com/ontologia/mochilas_voadoras#idMateriaPrima -- http://www.w3.org/1999/02/22-rdf-syntax-ns#type --> http://www.w3.org/2002/07/owl#DatatypeProperty
  http://www.exemplo.com/ontologia/mochilas_voadoras#temFrequenciaAtualizacao -- http://www.w3.org/2000/01/rdf-schema#range --> http://www.w3.org/2001/XMLSchema#string
  http://www.exemplo.com/ontologia/mochilas_voadoras#cidade -- http://www.w3.org/2000/01/rdf-schema#label --> cidade
  http://ww

In [11]:
# Célula 9: Limpar o Banco de Dados Neo4j
# Esta célula executa um comando Cypher para deletar todos os nós e relacionamentos existentes no banco Neo4j.
# Isso garante um ambiente limpo para a criação do novo grafo de conhecimento.
# CUIDADO: Este comando APAGA TODOS os dados do seu banco Neo4j conectado.
# Certifique-se de que você está conectado à instância correta do AuraDB e que não há dados importantes lá.

print("Limpeza do banco de dados Neo4j iniciada...")

# Verifica se o driver do Neo4j foi inicializado corretamente na Célula 5
if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou na Célula 5.")
else:
    try:
        # Abre uma sessão para interagir com o banco de dados
        with driver.session() as session:
            # Comando Cypher para deletar todos os nós e seus relacionamentos.
            # 'MATCH (n)' encontra todos os nós.
            # 'DETACH DELETE n' deleta os nós e quaisquer relacionamentos que eles possam ter.
            session.run("MATCH (n) DETACH DELETE n")
            print("Banco de dados Neo4j limpo com sucesso! ✅")
            print("Seu grafo está pronto para ser preenchido.")

    except Exception as e:
        print(f"ERRO ao limpar o banco de dados Neo4j: {e}")
        print("Verifique a conexão (Célula 5) e as permissões do usuário Neo4j.")

Limpeza do banco de dados Neo4j iniciada...
Banco de dados Neo4j limpo com sucesso! ✅
Seu grafo está pronto para ser preenchido.


In [12]:
# Célula 10: Criar Nodos para Tabelas e Colunas no Neo4j (COM EMBEDDINGS)
# Esta célula cria nós para tabelas e colunas no Neo4j e, crucialmente,
# calcula e armazena um embedding vetorial para cada nó, usando o modelo de embeddings da Célula 4.
# Isso é essencial para a busca de similaridade semântica posterior.

print("Iniciando a criação de nós para tabelas e colunas no Neo4j (com embeddings)...")

if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou anteriormente.")
elif modelo_logico_df is None:
    print("ERRO: O DataFrame 'modelo_logico_df' não foi carregado. Verifique a Célula 6.")
elif embeddings is None: # Verifica se o modelo de embeddings foi carregado
    print("ERRO: O modelo de Embeddings (Célula 4) não está disponível. Não é possível gerar embeddings para os nós.")
    raise ValueError("Modelo de Embeddings ausente.")
else:
    try:
        with driver.session() as session:
            created_tables = set()

            for index, row in modelo_logico_df.iterrows():
                table_name = row['Nome_Tabela']
                column_name = row['Nome_Coluna']
                data_type = row['Tipo_Dado']
                key_type = row['Chave']
                description = row['Descricao_Coluna']
                business_domain = row['Dominio_Negocio']

                # --- 1. Criar Nó para a Tabela e Adicionar Embedding ---
                if table_name not in created_tables:
                    table_text = f"Tabela: {table_name}. Domínio: {business_domain}."
                    table_embedding = embeddings.embed_query(table_text)

                    query_create_table = """
                    MERGE (t:Table {name: $table_name})
                    SET t.domain = $business_domain,
                        t.embedding = $table_embedding
                    RETURN t
                    """
                    session.run(query_create_table,
                                table_name=table_name,
                                business_domain=business_domain,
                                table_embedding=table_embedding)
                    created_tables.add(table_name)

                # --- 2. Criar Nó para a Coluna e Adicionar Embedding ---
                column_text = f"Coluna: {column_name}. Tabela: {table_name}. Tipo: {data_type}. Descrição: {description}. Domínio: {business_domain}."
                column_embedding = embeddings.embed_query(column_text)

                query_create_column = """
                MERGE (c:Column {name: $column_name, tableName: $table_name})
                SET c.dataType = $data_type,
                    c.keyType = $key_type,
                    c.description = $description,
                    c.domain = $business_domain,
                    c.embedding = $column_embedding
                RETURN c
                """
                session.run(query_create_column,
                            column_name=column_name,
                            table_name=table_name,
                            data_type=data_type,
                            key_type=key_type,
                            description=description,
                            business_domain=business_domain,
                            column_embedding=column_embedding)

            print("Criação de nós para tabelas e colunas (com embeddings) concluída com sucesso! ✅")
            print(f"Foram processadas {len(created_tables)} tabelas únicas e {len(modelo_logico_df)} colunas.")

            result_tables = session.run("MATCH (n:Table) RETURN count(n) AS num_tables, head(collect(n.embedding)) AS sample_embedding_table LIMIT 1").single()
            result_columns = session.run("MATCH (n:Column) RETURN count(n) AS num_columns, head(collect(n.embedding)) AS sample_embedding_column LIMIT 1").single()

            print(f"Confirmação no Neo4j: {result_tables['num_tables']} nós de Tabelas e {result_columns['num_columns']} nós de Colunas.")
            if result_tables['sample_embedding_table'] and result_columns['sample_embedding_column']:
                print(f"Embedding de Tabela de Exemplo (primeiros 5): {result_tables['sample_embedding_table'][:5]}...")
                print(f"Embedding de Coluna de Exemplo (primeiros 5): {result_columns['sample_embedding_column'][:5]}...")
                print(f"Tamanho do Embedding: {len(result_tables['sample_embedding_table'])}")
            else:
                print("Embeddings não encontrados em nós de exemplo (verifique a Célula 4).")


    except Exception as e:
        print(f"ERRO ao criar nós (com embeddings) no Neo4j: {e}")
        print("Verifique se o driver está ativo e se o modelo de embeddings (Célula 4) foi carregado.")

Iniciando a criação de nós para tabelas e colunas no Neo4j (com embeddings)...
Criação de nós para tabelas e colunas (com embeddings) concluída com sucesso! ✅
Foram processadas 28 tabelas únicas e 109 colunas.
Confirmação no Neo4j: 28 nós de Tabelas e 109 nós de Colunas.
Embedding de Tabela de Exemplo (primeiros 5): [0.11474821716547012, 0.11224862188100815, -0.16205886006355286, -0.5270341038703918, 0.035828351974487305]...
Embedding de Coluna de Exemplo (primeiros 5): [-0.16314740478992462, -0.04211324825882912, -0.2855185568332672, 0.1186695545911789, -0.00236457004211843]...
Tamanho do Embedding: 384


In [13]:
# Célula 11: Criar Relacionamentos PK-FK entre Tabelas no Neo4j
# Esta célula itera sobre o DataFrame do modelo lógico para identificar colunas que são chaves estrangeiras (FKs)
# e cria relacionamentos do tipo ':HAS_FK_TO' entre os nós de Tabela de origem e de destino.
# Além disso, cria um relacionamento ':HAS_COLUMN' entre a Tabela e suas Colunas.

print("Iniciando a criação de relacionamentos no Neo4j...")

if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou anteriormente.")
elif modelo_logico_df is None:
    print("ERRO: O DataFrame 'modelo_logico_df' não foi carregado. Verifique a Célula 6.")
else:
    try:
        with driver.session() as session:
            # 1. Criar relacionamentos entre Tabelas e suas Colunas
            print("Criando relacionamentos :HAS_COLUMN entre Tabelas e Colunas...")
            query_has_column = """
            MATCH (t:Table {name: $table_name})
            MATCH (c:Column {name: $column_name, tableName: $table_name})
            MERGE (t)-[:HAS_COLUMN]->(c)
            """
            for index, row in modelo_logico_df.iterrows():
                session.run(query_has_column,
                            table_name=row['Nome_Tabela'],
                            column_name=row['Nome_Coluna'])
            print("Relacionamentos :HAS_COLUMN criados com sucesso.")

            # 2. Criar relacionamentos PK-FK entre Tabelas
            print("Criando relacionamentos :HAS_FK_TO entre Tabelas via chaves estrangeiras...")

            # Filtra o DataFrame para obter apenas as colunas que são chaves estrangeiras (FK)
            fk_columns_df = modelo_logico_df[modelo_logico_df['Chave'] == 'FK']

            for index, row in fk_columns_df.iterrows():
                source_table = row['Nome_Tabela']
                fk_column = row['Nome_Coluna']
                target_table = row['Referencia_Tabela_FK']
                target_pk_column = row['Referencia_Coluna_FK']

                # Consulta Cypher para criar o relacionamento PK-FK
                # Encontra a tabela de origem (source_table)
                # Encontra a coluna FK na tabela de origem (fk_column)
                # Encontra a tabela de destino (target_table)
                # Encontra a coluna PK na tabela de destino (target_pk_column)
                # Cria um relacionamento :HAS_FK_TO da tabela de origem para a tabela de destino,
                # e um relacionamento :REFERENCES da coluna FK para a coluna PK referenciada.
                query_create_fk_relationship = """
                MATCH (source:Table {name: $source_table})
                MATCH (target:Table {name: $target_table})
                MATCH (fk_col:Column {name: $fk_column, tableName: $source_table})
                MATCH (pk_col:Column {name: $target_pk_column, tableName: $target_table})

                MERGE (source)-[r:HAS_FK_TO]->(target)
                SET r.fk_column = $fk_column, r.pk_column = $target_pk_column

                MERGE (fk_col)-[:REFERENCES]->(pk_col)
                """
                session.run(query_create_fk_relationship,
                            source_table=source_table,
                            target_table=target_table,
                            fk_column=fk_column,
                            target_pk_column=target_pk_column)

            print("Criação de relacionamentos PK-FK e HAS_COLUMN concluída com sucesso! ✅")

            # Verificação rápida no Neo4j: conta os relacionamentos
            result_fk_rels = session.run("MATCH ()-[r:HAS_FK_TO]->() RETURN count(r) AS num_fk_rels").single()["num_fk_rels"]
            result_has_column_rels = session.run("MATCH ()-[r:HAS_COLUMN]->() RETURN count(r) AS num_has_column_rels").single()["num_has_column_rels"]
            result_references_rels = session.run("MATCH ()-[r:REFERENCES]->() RETURN count(r) AS num_references_rels").single()["num_references_rels"]

            print(f"Confirmação no Neo4j: {result_fk_rels} relacionamentos :HAS_FK_TO, {result_has_column_rels} relacionamentos :HAS_COLUMN, e {result_references_rels} relacionamentos :REFERENCES criados.")

    except Exception as e:
        print(f"ERRO ao criar relacionamentos no Neo4j: {e}")
        print("Verifique os dados do DataFrame, especialmente as colunas de FKs e suas referências.")

Iniciando a criação de relacionamentos no Neo4j...
Criando relacionamentos :HAS_COLUMN entre Tabelas e Colunas...
Relacionamentos :HAS_COLUMN criados com sucesso.
Criando relacionamentos :HAS_FK_TO entre Tabelas via chaves estrangeiras...
Criação de relacionamentos PK-FK e HAS_COLUMN concluída com sucesso! ✅
Confirmação no Neo4j: 23 relacionamentos :HAS_FK_TO, 109 relacionamentos :HAS_COLUMN, e 23 relacionamentos :REFERENCES criados.


In [14]:
# Célula 12: Adicionar Metadados de Governança aos Nodos de Coluna
# Esta célula itera sobre o DataFrame de metadados de governança
# e adiciona propriedades adicionais (metadados DMBOK) aos nós de 'Column' existentes no Neo4j.
# Isso torna seu grafo mais rico em contexto e governança.

print("Iniciando a adição de metadados de governança aos nós de Coluna no Neo4j...")

if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou anteriormente.")
elif metadados_governanca_df is None:
    print("ERRO: O DataFrame 'metadados_governanca_df' não foi carregado. Verifique a Célula 7.")
else:
    try:
        with driver.session() as session:
            # Itera sobre cada linha do DataFrame de metadados de governança
            for index, row in metadados_governanca_df.iterrows():
                table_name = row['Nome_Tabela']
                column_name = row['Nome_Coluna']

                # Coleta todos os metadados de governança relevantes.
                # Usamos .get(column, '') para lidar com colunas que podem não existir ou serem NaN no DF,
                # convertendo para string vazia para evitar problemas no Cypher.
                classificacao_confidencialidade = row.get('Classificacao_Confidencialidade', '')
                proprietario_dado = row.get('Proprietario_Dado', '')
                regras_qualidade_dados = row.get('Regras_Qualidade_Dados', '')
                freq_atualizacao = row.get('Freq_Atualizacao', '')
                retencao_dados_anos = row.get('Retencao_Dados_Anos', '')
                regra_negocio_associada = row.get('Regra_Negocio_Associada', '')
                termo_glossario_dmbok = row.get('Termo_Glossario_DMBOK', '')

                # Consulta Cypher para encontrar o nó da Coluna e adicionar/atualizar suas propriedades.
                # MERGE (c:Column {name: $column_name, tableName: $table_name}) garante que o nó exista.
                # SET c.propriedade = $valor_propriedade adiciona/atualiza as propriedades.
                query_add_metadata = """
                MATCH (c:Column {name: $column_name, tableName: $table_name})
                SET c.ClassificacaoConfidencialidade = $classificacao_confidencialidade,
                    c.ProprietarioDado = $proprietario_dado,
                    c.RegrasQualidadeDados = $regras_qualidade_dados,
                    c.FreqAtualizacao = $freq_atualizacao,
                    c.RetencaoDadosAnos = $retencao_dados_anos,
                    c.RegraNegocioAssociada = $regra_negocio_associada,
                    c.TermoGlossarioDMBOK = $termo_glossario_dmbok
                RETURN c
                """
                session.run(query_add_metadata,
                            column_name=column_name,
                            table_name=table_name,
                            classificacao_confidencialidade=classificacao_confidencialidade,
                            proprietario_dado=proprietario_dado,
                            regras_qualidade_dados=regras_qualidade_dados,
                            freq_atualizacao=freq_atualizacao,
                            retencao_dados_anos=retencao_dados_anos,
                            regra_negocio_associada=regra_negocio_associada,
                            termo_glossario_dmbok=termo_glossario_dmbok)

            print("Metadados de governança adicionados aos nós de Coluna com sucesso! ✅")

            # Verificação rápida no Neo4j: Exemplo de uma coluna com metadados
            print("\nVerificando metadados de uma coluna de exemplo (Cliente.Email_Cliente):")
            sample_query = """
            MATCH (c:Column {name: 'Email_Cliente', tableName: 'Cliente'})
            RETURN c.ClassificacaoConfidencialidade AS Confidencialidade,
                   c.ProprietarioDado AS Proprietario,
                   c.RegrasQualidadeDados AS Qualidade,
                   c.RetencaoDadosAnos AS Retencao
            """
            result = session.run(sample_query).single()
            if result:
                print(result.data())
            else:
                print("Coluna de exemplo não encontrada ou sem metadados. Verifique se o nome da coluna está correto.")

    except Exception as e:
        print(f"ERRO ao adicionar metadados no Neo4j: {e}")
        print("Verifique se os nós de Coluna foram criados (Célula 10) e se o DataFrame de metadados está correto.")

Iniciando a adição de metadados de governança aos nós de Coluna no Neo4j...
Metadados de governança adicionados aos nós de Coluna com sucesso! ✅

Verificando metadados de uma coluna de exemplo (Cliente.Email_Cliente):
{'Confidencialidade': 'Restrito', 'Proprietario': 'Marketing', 'Qualidade': 'Não nulo; formato de e-mail válido; único', 'Retencao': 10}


In [15]:
# Célula 13: Injetar a Ontologia no Grafo do Neo4j (COM EMBEDDINGS)
# Esta célula injeta classes, propriedades da ontologia e, crucialmente,
# calcula e armazena um embedding vetorial para cada nó de Conceito e Propriedade.

print("Iniciando a injeção da ontologia no grafo do Neo4j (com embeddings)...")

from rdflib import Graph, URIRef, Literal, Namespace, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD

if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou na Célula 5.")
elif 'ontology_graph' not in locals() or ontology_graph is None:
    print("ERRO: O grafo 'ontology_graph' não foi carregado. Verifique a Célula 8.")
elif embeddings is None:
    print("ERRO: O modelo de Embeddings (Célula 4) não está disponível. Não é possível gerar embeddings para a ontologia.")
    raise ValueError("Modelo de Embeddings ausente.")
else:
    try:
        with driver.session() as session:
            def uri_to_name(uri):
                s_uri = str(uri)
                if '#' in s_uri:
                    return s_uri.split('#')[-1]
                elif '/' in s_uri:
                    return s_uri.split('/')[-1]
                return s_uri

            # 1. Injetar Classes (Conceitos) como nós :Concept e Adicionar Embedding
            print("Injetando classes da ontologia como nós :Concept (com embeddings)...")
            for s, p, o in ontology_graph.triples((None, RDF.type, OWL.Class)):
                concept_name = uri_to_name(s)
                comment_triples = list(ontology_graph.triples((s, RDFS.comment, None)))
                concept_text = f"Conceito: {concept_name}."
                if comment_triples:
                    concept_text += f" Descrição: {str(comment_triples[0][2])}"
                concept_embedding = embeddings.embed_query(concept_text)

                query_create_concept = """
                MERGE (c:Concept {name: $concept_name})
                SET c.uri = $concept_uri,
                    c.embedding = $concept_embedding
                """
                session.run(query_create_concept, concept_name=concept_name, concept_uri=str(s), concept_embedding=concept_embedding)
            print("Classes injetadas.")

            # 2. Injetar Relações de Hierarquia (SubClasses - IS_A)
            print("Injetando relações de hierarquia (subClasses - IS_A)...")
            for s, p, o in ontology_graph.triples((None, RDFS.subClassOf, None)):
                if isinstance(o, URIRef):
                    sub_class_name = uri_to_name(s)
                    super_class_name = uri_to_name(o)
                    query_create_subclass_rel = """
                    MATCH (sub:Concept {name: $sub_class_name})
                    MATCH (super:Concept {name: $super_class_name})
                    MERGE (sub)-[:IS_A]->(super)
                    """
                    session.run(query_create_subclass_rel,
                                sub_class_name=sub_class_name,
                                super_class_name=super_class_name)
            print("Relações de hierarquia injetadas.")

            # 3. Injetar Propriedades (ObjectProperties e DatatypeProperties) como nós :Property e Adicionar Embedding
            print("Injetando Propriedades (ObjectProperties e DatatypeProperties - com embeddings)...")
            for s, p, o in ontology_graph.triples((None, RDF.type, None)):
                if o == OWL.ObjectProperty or o == OWL.DatatypeProperty:
                    prop_name = uri_to_name(s)
                    prop_type = "ObjectProperty" if o == OWL.ObjectProperty else "DatatypeProperty"

                    comment_triples = list(ontology_graph.triples((s, RDFS.comment, None)))
                    prop_text = f"Propriedade: {prop_name}. Tipo: {prop_type}."
                    if comment_triples:
                        prop_text += f" Descrição: {str(comment_triples[0][2])}"
                    prop_embedding = embeddings.embed_query(prop_text)

                    query_create_property = """
                    MERGE (p:Property {name: $prop_name})
                    SET p.type = $prop_type, p.uri = $prop_uri,
                        p.embedding = $prop_embedding
                    """
                    session.run(query_create_property, prop_name=prop_name, prop_type=prop_type, prop_uri=str(s), prop_embedding=prop_embedding)

                    for _, _, domain in ontology_graph.triples((s, RDFS.domain, None)):
                        domain_name = uri_to_name(domain)
                        query_prop_domain = """
                        MATCH (p:Property {name: $prop_name})
                        MATCH (c:Concept {name: $domain_name})
                        MERGE (p)-[:HAS_DOMAIN]->(c)
                        """
                        session.run(query_prop_domain, prop_name=prop_name, domain_name=domain_name)

                    for _, _, range_val in ontology_graph.triples((s, RDFS.range, None)):
                        range_name = uri_to_name(range_val)
                        if str(range_val).startswith(str(XSD)):
                            query_prop_range = """
                            MATCH (p:Property {name: $prop_name})
                            MERGE (dt:DataType {name: $range_name})
                            MERGE (p)-[:HAS_RANGE]->(dt)
                            """
                        else:
                            query_prop_range = """
                            MATCH (p:Property {name: $prop_name})
                            MATCH (c:Concept {name: $range_name})
                            MERGE (p)-[:HAS_RANGE]->(c)
                            """
                        session.run(query_prop_range, prop_name=prop_name, range_name=range_name)
            print("Propriedades injetadas.")

            # 4. Injetar Comentários e Labels Adicionais (sinônimos)
            print("Injetando comentários e labels adicionais...")
            for s, p, o in ontology_graph.triples((None, RDFS.comment, None)):
                node_name = uri_to_name(s)
                query_add_comment = """
                MATCH (n) WHERE n.name = $node_name
                SET n.comment = $comment_text
                """
                session.run(query_add_comment, node_name=node_name, comment_text=str(o))

            for s, p, o in ontology_graph.triples((None, RDFS.label, None)):
                node_name = uri_to_name(s)
                if node_name.lower() != str(o).lower():
                    query_add_label = """
                    MATCH (n) WHERE n.name = $node_name
                    SET n.alt_labels = coalesce(n.alt_labels, []) + $label_text
                    """
                    session.run(query_add_label, node_name=node_name, label_text=str(o))
            print("Comentários e labels adicionais injetados.")

            # 5. Injetar Restrições/Axiomas Básicos (para cardinalidade e relações inferidas)
            print("Injetando restrições e axiomas básicos (simplificado)...")
            for s, p, o in ontology_graph.triples((None, RDFS.subClassOf, None)):
                if isinstance(o, BNode):
                    for _, prop_on, prop_name_uri in ontology_graph.triples((o, OWL.onProperty, None)):
                        prop_name = uri_to_name(prop_name_uri)

                        if (o, OWL.minQualifiedCardinality, None) in ontology_graph:
                            cardinality_val = ontology_graph.value(o, OWL.minQualifiedCardinality)
                            target_class_uri = ontology_graph.value(o, OWL.onClass)
                            if target_class_uri:
                                target_class_name = uri_to_name(target_class_uri)
                                query_add_restriction = """
                                MATCH (c:Concept {name: $class_name})
                                MATCH (p:Property {name: $prop_name})
                                MATCH (tc:Concept {name: $target_class_name})
                                MERGE (c)-[:HAS_RESTRICTION]->(p)
                                SET p.minCardinality = $card_val, p.onClass = $target_class_name
                                """
                                session.run(query_add_restriction, class_name=uri_to_name(s), prop_name=prop_name, card_val=str(cardinality_val), target_class_name=target_class_name)

            print("Restrições e axiomas injetados (simplificado).")

            print("Injeção da ontologia no Neo4j (com embeddings) concluída com sucesso! ✅")

            num_concepts = session.run("MATCH (n:Concept) RETURN count(n) AS num, head(collect(n.embedding)) AS sample_embedding_concept LIMIT 1").single()
            num_properties = session.run("MATCH (n:Property) RETURN count(n) AS num, head(collect(n.embedding)) AS sample_embedding_property LIMIT 1").single()

            print(f"Confirmação no Neo4j: {num_concepts['num']} nós de Conceito, {num_properties['num']} nós de Propriedade.")
            if num_concepts['sample_embedding_concept'] and num_properties['sample_embedding_property']:
                print(f"Embedding de Conceito de Exemplo (primeiros 5): {num_concepts['sample_embedding_concept'][:5]}...")
                print(f"Embedding de Propriedade de Exemplo (primeiros 5): {num_properties['sample_embedding_property'][:5]}...")
            else:
                print("Embeddings não encontrados em nós de Conceito/Propriedade de exemplo (verifique a Célula 4).")


    except Exception as e:
        print(f"ERRO ao injetar a ontologia (com embeddings) no Neo4j: {e}")
        print("Verifique a sintaxe da sua ontologia (arquivo TTL) e a conexão com o Neo4j.")
        print("Detalhes do erro:", e)

Iniciando a injeção da ontologia no grafo do Neo4j (com embeddings)...
Injetando classes da ontologia como nós :Concept (com embeddings)...
Classes injetadas.
Injetando relações de hierarquia (subClasses - IS_A)...
Relações de hierarquia injetadas.
Injetando Propriedades (ObjectProperties e DatatypeProperties - com embeddings)...
Propriedades injetadas.
Injetando comentários e labels adicionais...
Comentários e labels adicionais injetados.
Injetando restrições e axiomas básicos (simplificado)...
Restrições e axiomas injetados (simplificado).
Injeção da ontologia no Neo4j (com embeddings) concluída com sucesso! ✅
Confirmação no Neo4j: 22 nós de Conceito, 119 nós de Propriedade.
Embedding de Conceito de Exemplo (primeiros 5): [-0.13235929608345032, 0.005861241836100817, -0.36951690912246704, -0.44113877415657043, -0.18100114166736603]...
Embedding de Propriedade de Exemplo (primeiros 5): [-0.2967492640018463, -0.1189851313829422, -0.4497503638267517, -0.4096130132675171, -0.2168119549751

In [16]:
# Célula 14: Criar Dados de Exemplo no Neo4j
# Esta célula insere alguns nós de dados de exemplo (instâncias)
# para as classes principais. Labels específicos são adicionados estaticamente.

print("Iniciando a criação de dados de exemplo no Neo4j...")

if driver is None:
    print("ERRO: O driver do Neo4j não está disponível. Conexão falhou anteriormente.")
else:
    try:
        with driver.session() as session:
            # Dados de exemplo para Cliente (Concessionárias)
            clientes_exemplo = [
                {"id": 101, "nome": "Aventura Extrema Ltda.", "email": "contato@aventuraextrema.com", "cnpj": "11.222.333/0001-44"},
                {"id": 102, "nome": "Rodas & Asas S.A.", "email": "vendas@rodaseasas.net", "cnpj": "22.333.444/0001-55"},
                {"id": 103, "nome": "Trilhas Radicais ME", "email": "info@trilhasradicais.com.br", "cnpj": "33.444.555/0001-66"}
            ]
            query_create_clientes = """
            UNWIND $clientes AS cliente_data
            MERGE (c:Cliente {ID_Cliente: cliente_data.id})
            SET c.Nome_Cliente = cliente_data.nome,
                c.Email_Cliente = cliente_data.email,
                c.CNPJ = cliente_data.cnpj
            SET c:Concessionaria, c:ClienteB2B
            """
            session.run(query_create_clientes, clientes=clientes_exemplo)
            print(f"  {len(clientes_exemplo)} Clientes de exemplo criados/verificados.")

            # Dados de exemplo para Produto
            produtos_exemplo = [
                {"id": 201, "nome": "Quadriciclo X-Treme", "descricao": "Veículo off-road de alta performance.", "preco": 35000.00, "categoria_id": 1},
                {"id": 202, "nome": "Mochila Voadora AeroPack", "descricao": "Mochila pessoal com propulsão a jato.", "preco": 120000.00, "categoria_id": 2},
                {"id": 203, "nome": "Bike Montanha Carbono", "descricao": "Bicicleta leve para trilhas agressivas.", "preco": 8000.00, "categoria_id": 1}
            ]
            query_create_produtos = """
            UNWIND $produtos AS produto_data
            MERGE (p:Produto {ID_Produto: produto_data.id})
            SET p.Nome_Produto = produto_data.nome,
                p.Descricao_Produto = produto_data.descricao,
                p.Preco_Unitario_Venda = produto_data.preco
            WITH p, produto_data
            // Adiciona o label específico do tipo de produto da ontologia (manualmente, sem APOC)
            // Aqui, o LLM precisará ser instruído a buscar 'Mochila Voadora' na propriedade p.Nome_Produto.

            MERGE (cat:Categoria_Produto {ID_Categoria_Produto: produto_data.categoria_id})
            ON CREATE SET cat.Nome_Categoria = CASE WHEN produto_data.categoria_id = 1 THEN 'Veículos' WHEN produto_data.categoria_id = 2 THEN 'Equipamentos Voadores' ELSE 'Outros' END
            MERGE (p)-[:PERTENCE_A_CATEGORIA]->(cat)
            """
            session.run(query_create_produtos, produtos=produtos_exemplo)
            print(f"  {len(produtos_exemplo)} Produtos de exemplo e suas categorias criados/verificados.")

            # Dados de exemplo para Pedido e Item_Pedido
            pedidos_exemplo = [
                {"id": 301, "cliente_id": 101, "data": "2024-05-10T10:00:00", "status": "Enviado",
                 "itens": [{"produto_id": 201, "quantidade": 2, "preco_venda": 35000.00, "item_id": 1}]},
                {"id": 302, "cliente_id": 102, "data": "2024-05-15T14:30:00", "status": "Pendente",
                 "itens": [{"produto_id": 202, "quantidade": 1, "preco_venda": 120000.00, "item_id": 2}]},
                {"id": 303, "cliente_id": 101, "data": "2024-06-01T09:00:00", "status": "Processando",
                 "itens": [{"produto_id": 203, "quantidade": 3, "preco_venda": 8000.00, "item_id": 3},
                           {"produto_id": 201, "quantidade": 1, "preco_venda": 35000.00, "item_id": 4}]}
            ]
            query_create_pedidos = """
            UNWIND $pedidos AS pedido_data
            MERGE (ped:Pedido {ID_Pedido: pedido_data.id})
            SET ped.Data_Pedido = pedido_data.data,
                ped.Status_Pedido = pedido_data.status
            WITH ped, pedido_data
            MATCH (cli:Cliente {ID_Cliente: pedido_data.cliente_id})
            MERGE (cli)-[:REALIZOU_PEDIDO]->(ped)
            WITH ped, pedido_data
            UNWIND pedido_data.itens AS item_data
            MERGE (item:Item_Pedido {ID_Item_Pedido: item_data.item_id}) // Usando ID numérico aqui
            SET item.Quantidade = item_data.quantidade,
                item.Preco_Venda = item_data.preco_venda
            WITH ped, item, item_data
            MATCH (prod:Produto {ID_Produto: item_data.produto_id})
            MERGE (ped)-[:CONTEM_ITEM]->(item)
            MERGE (item)-[:REFERENCIA_PRODUTO]->(prod)
            """
            session.run(query_create_pedidos, pedidos=pedidos_exemplo)
            print(f"  {len(pedidos_exemplo)} Pedidos de exemplo e seus itens criados/verificados.")

            # Dados de exemplo para Funcionário
            funcionarios_exemplo = [
                {"id": 401, "nome": "Ana Silva", "email": "ana.silva@empresa.com", "data_contratacao": "2020-01-15", "cargo_id": 1001, "depto_id": 10},
                {"id": 402, "nome": "Bruno Costa", "email": "bruno.costa@empresa.com", "data_contratacao": "2021-03-20", "cargo_id": 1002, "depto_id": 20},
                {"id": 403, "nome": "Carlos Rocha", "email": "carlos.rocha@empresa.com", "data_contratacao": "2019-07-01", "cargo_id": 1003, "depto_id": 30}
            ]
            query_create_funcionarios = """
            UNWIND $funcionarios AS func_data
            MERGE (f:Funcionario {ID_Funcionario: func_data.id})
            SET f.Nome_Funcionario = func_data.nome,
                f.Email_Funcionario = func_data.email,
                f.Data_Contratacao = func_data.data_contratacao
            WITH f, func_data
            MERGE (c:Cargo {ID_Cargo: func_data.cargo_id})
            ON CREATE SET c.Nome_Cargo = CASE WHEN func_data.cargo_id = 1001 THEN 'Gerente de Vendas' WHEN func_data.cargo_id = 1002 THEN 'Analista de TI' WHEN func_data.cargo_id = 1003 THEN 'Diretor de Produção' ELSE 'Outro Cargo' END
            MERGE (f)-[:TEM_CARGO]->(c)
            WITH f, func_data
            MERGE (d:Departamento {ID_Departamento: func_data.depto_id})
            ON CREATE SET d.Nome_Departamento = CASE WHEN func_data.depto_id = 10 THEN 'Vendas' WHEN func_data.depto_id = 20 THEN 'TI' WHEN func_data.depto_id = 30 THEN 'Produção' ELSE 'Outro Departamento' END
            MERGE (f)-[:PERTENCE_A_DEPARTAMENTO]->(d)
            """
            session.run(query_create_funcionarios, funcionarios=funcionarios_exemplo)
            print(f"  {len(funcionarios_exemplo)} Funcionários de exemplo criados/verificados.")


            print("Criação de dados de exemplo no Neo4j concluída com sucesso! ✅")

            num_clientes = session.run("MATCH (n:Cliente) RETURN count(n) AS num").single()["num"]
            num_produtos = session.run("MATCH (n:Produto) RETURN count(n) AS num").single()["num"]
            num_pedidos = session.run("MATCH (n:Pedido) RETURN count(n) AS num").single()["num"]
            num_funcionarios = session.run("MATCH (n:Funcionario) RETURN count(n) AS num").single()["num"]

            print(f"Confirmação no Neo4j: {num_clientes} Clientes, {num_produtos} Produtos, {num_pedidos} Pedidos, {num_funcionarios} Funcionários de exemplo.")

    except Exception as e:
        print(f"ERRO ao criar dados de exemplo no Neo4j: {e}")
        print("Verifique a conexão e as queries Cypher para os dados de exemplo.")
        print("Detalhes do erro:", e)

Iniciando a criação de dados de exemplo no Neo4j...
  3 Clientes de exemplo criados/verificados.
  3 Produtos de exemplo e suas categorias criados/verificados.
  3 Pedidos de exemplo e seus itens criados/verificados.
  3 Funcionários de exemplo criados/verificados.
Criação de dados de exemplo no Neo4j concluída com sucesso! ✅
Confirmação no Neo4j: 3 Clientes, 3 Produtos, 3 Pedidos, 3 Funcionários de exemplo.


In [17]:
# Célula 15: Função de Consulta ao Grafo
# Esta célula define uma função Python para executar consultas Cypher no banco Neo4j.

print("Definindo função de consulta ao grafo Neo4j...")

def run_cypher_query(query, parameters=None):
    """
    Executa uma consulta Cypher no banco de dados Neo4j.

    Args:
        query (str): A string da consulta Cypher a ser executada.
        parameters (dict, optional): Um dicionário de parâmetros para a consulta. Padrão para None.

    Returns:
        list: Uma lista de dicionários, onde cada dicionário representa um registro do resultado da consulta.
              Retorna uma lista vazia se não houver resultados.
    """
    if driver is None:
        print("Erro: Driver do Neo4j não está disponível.")
        return []

    results = []
    try:
        with driver.session() as session:
            result = session.run(query, parameters)
            for record in result:
                results.append(record.data())
        # print(f"Consulta Cypher executada com sucesso. Retornou {len(results)} registros. ✅") # Comentado para evitar spam
        return results

    except Exception as e:
        print(f"ERRO ao executar consulta Cypher: {e}")
        return []

print("Função 'run_cypher_query' definida. ✅")

# Teste rápido da função: Contar todos os nós no grafo
print("\nTestando a função de consulta: Contando todos os nós...")
test_query_all_nodes = "MATCH (n) RETURN count(n) AS totalNodes"
node_count_result = run_cypher_query(test_query_all_nodes)

if node_count_result:
    print(f"Total de nós no grafo (confirmado pela função): {node_count_result[0]['totalNodes']}")
else:
    print("Não foi possível contar os nós. Algo está errado com a função de consulta ou com a conexão.")

# Teste rápido da função: Buscar um produto de exemplo
print("\nTestando a função de consulta: Buscando um produto de exemplo...")
test_query_product = "MATCH (p:Produto {Nome_Produto: 'Quadriciclo X-Treme'}) RETURN p.Nome_Produto AS Nome, p.Preco_Unitario_Venda AS Preco"
product_result = run_cypher_query(test_query_product)

if product_result:
    print(f"Produto encontrado: {product_result}")
else:
    print("Produto de exemplo 'Quadriciclo X-Treme' não encontrado.")

Definindo função de consulta ao grafo Neo4j...
Função 'run_cypher_query' definida. ✅

Testando a função de consulta: Contando todos os nós...
Total de nós no grafo (confirmado pela função): 307

Testando a função de consulta: Buscando um produto de exemplo...
Produto encontrado: [{'Nome': 'Quadriciclo X-Treme', 'Preco': 35000.0}]


In [18]:
# Célula 16: Criar e Popular Banco de Dados SQLite
# Esta célula cria um banco de dados SQLite espelhando o modelo lógico.
# A criação de tabelas e a inserção de dados foram robustecidas para lidar com nuances do SQLite,
# garantindo que a população ocorra sem erros de tipo.

import sqlite3
import pandas as pd
import os

print("Iniciando a criação e população do banco de dados SQLite...")

sqlite_db_path = '/content/drive/MyDrive/00_Projeto_ICMC/database.db' # <--- AJUSTE O CAMINHO SE NECESSÁRIO
os.makedirs(os.path.dirname(sqlite_db_path), exist_ok=True)

conn = None
cursor = None

try:
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    # --- 1. Excluir tabelas existentes (para garantir um ambiente limpo para o SQLite) ---
    print("  Removendo tabelas SQLite existentes (se houver)...")
    all_tables = modelo_logico_df['Nome_Tabela'].unique().tolist()
    for table_name in reversed(all_tables):
        try:
            cursor.execute(f"DROP TABLE IF EXISTS {table_name}")
        except sqlite3.OperationalError as e:
            print(f"    Aviso: Não foi possível dropar a tabela {table_name}: {e}")
    conn.commit()
    print("  Tabelas SQLite limpas.")

    # --- 2. Gerar e Executar CREATE TABLE statements ---
    print("  Criando tabelas no SQLite com base no modelo lógico...")

    type_mapping = {
        'INT': 'INTEGER',
        'VARCHAR(255)': 'TEXT', 'VARCHAR(100)': 'TEXT', 'VARCHAR(50)': 'TEXT',
        'VARCHAR(20)': 'TEXT', 'VARCHAR(10)': 'TEXT', 'TEXT': 'TEXT',
        'DATE': 'TEXT', 'DATETIME': 'TEXT',
        'DECIMAL(10,2)': 'REAL', 'DECIMAL(15,2)': 'REAL'
    }

    grouped_columns = modelo_logico_df.groupby('Nome_Tabela')

    for table_name, columns_df in grouped_columns:
        create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (\n"

        column_defs = []
        pk_cols = []
        fk_defs = []

        for _, col_row in columns_df.iterrows():
            col_name = col_row['Nome_Coluna']
            original_col_type = col_row['Tipo_Dado']
            key_type = col_row['Chave']

            col_type = type_mapping.get(original_col_type, 'TEXT')
            if key_type == 'PK' and 'INT' in original_col_type:
                col_type = 'INTEGER'
            elif key_type == 'PK' and 'VARCHAR' in original_col_type:
                col_type = 'TEXT'

            col_def = f"  {col_name} {col_type}"
            if key_type == 'PK':
                col_def += " PRIMARY KEY"
                pk_cols.append(col_name)
            elif key_type == 'NK':
                col_def += " UNIQUE"

            column_defs.append(col_def)

            if key_type == 'FK':
                ref_table = col_row['Referencia_Tabela_FK']
                ref_col = col_row['Referencia_Coluna_FK']
                fk_defs.append(f"  FOREIGN KEY ({col_name}) REFERENCES {ref_table}({ref_col})")

        create_table_sql += ",\n".join(column_defs)
        if fk_defs:
            create_table_sql += ",\n" + ",\n".join(fk_defs)

        create_table_sql += "\n);"
        cursor.execute(create_table_sql)
    conn.commit()
    print("  Tabelas SQLite criadas com sucesso.")

    # --- 3. Popular Tabelas com Dados de Exemplo (USANDO PARÂMETROS) ---
    print("  Populando tabelas SQLite com dados de exemplo...")

    # Cliente
    print("  --> Inserindo dados na tabela Cliente...")
    clientes_data = [(101, 'Aventura Extrema Ltda.', 'contato@aventuraextrema.com', '2023-01-15'),
                     (102, 'Rodas & Asas S.A.', 'vendas@rodaseasas.net', '2023-02-20'),
                     (103, 'Trilhas Radicais ME', 'info@trilhasradicais.com.br', '2023-03-01')]
    cursor.executemany("INSERT INTO Cliente (ID_Cliente, Nome_Cliente, Email_Cliente, Data_Cadastro) VALUES (?, ?, ?, ?)", clientes_data)

    # Endereco_Cliente
    print("  --> Inserindo dados na tabela Endereco_Cliente...")
    enderecos_data = [(1, 101, 'Rua das Trilhas', '123', 'Loja A', 'Centro', 'São Paulo', 'SP', '01000-000'),
                      (2, 102, 'Av. das Asas', '456', None, 'Aeroporto', 'Rio de Janeiro', 'RJ', '20000-000')]
    cursor.executemany("INSERT INTO Endereco_Cliente (ID_Endereco, ID_Cliente, Logradouro, Numero, Complemento, Bairro, Cidade, Estado, CEP) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", enderecos_data)

    # Categoria_Produto
    print("  --> Inserindo dados na tabela Categoria_Produto...")
    categorias_data = [(1, 'Veículos'), (2, 'Equipamentos Voadores'), (3, 'Acessórios')]
    cursor.executemany("INSERT INTO Categoria_Produto (ID_Categoria_Produto, Nome_Categoria) VALUES (?, ?)", categorias_data)

    # Produto
    print("  --> Inserindo dados na tabela Produto...")
    produtos_data = [(201, 'Quadriciclo X-Treme', 'Veículo off-road de alta performance.', 35000.00, 1),
                     (202, 'Mochila Voadora AeroPack', 'Mochila pessoal com propulsão a jato.', 120000.00, 2),
                     (203, 'Bike Montanha Carbono', 'Bicicleta leve para trilhas agressivas.', 8000.00, 1),
                     (204, 'Capacete Adventure Pro', 'Capacete com sistema de comunicação.', 1500.00, 3)]
    cursor.executemany("INSERT INTO Produto (ID_Produto, Nome_Produto, Descricao_Produto, Preco_Unitario_Venda, ID_Categoria_Produto) VALUES (?, ?, ?, ?, ?)", produtos_data)

    # Pedido
    print("  --> Inserindo dados na tabela Pedido...")
    pedidos_data = [(301, 101, '2024-05-10 10:00:00', 'Enviado'),
                    (302, 102, '2024-05-15 14:30:00', 'Pendente'),
                    (303, 101, '2024-06-01 09:00:00', 'Processando')]
    cursor.executemany("INSERT INTO Pedido (ID_Pedido, ID_Cliente, Data_Pedido, Status_Pedido) VALUES (?, ?, ?, ?)", pedidos_data)

    # Item_Pedido
    print("  --> Inserindo dados na tabela Item_Pedido...")
    itens_pedido_data = [(1, 301, 201, 2, 35000.00),
                         (2, 302, 202, 1, 120000.00),
                         (3, 303, 203, 3, 8000.00),
                         (4, 303, 201, 1, 35000.00)]
    cursor.executemany("INSERT INTO Item_Pedido (ID_Item_Pedido, ID_Pedido, ID_Produto, Quantidade, Preco_Venda) VALUES (?, ?, ?, ?, ?)", itens_pedido_data)

    # Estoque
    print("  --> Inserindo dados na tabela Estoque...")
    estoque_data = [(1, 201, 50, 'Galpao A'), (2, 202, 10, 'Galpao B')]
    cursor.executemany("INSERT INTO Estoque (ID_Estoque, ID_Produto, Quantidade_Disponivel, Local_Armazenamento) VALUES (?, ?, ?, ?)", estoque_data)

    # Fornecedor
    print("  --> Inserindo dados na tabela Fornecedor...")
    fornecedores_data = [(501, 'Metais Prime', 'contato@metaisprime.com'),
                         (502, 'Componentes Eletronicos S.A.', 'vendas@componentes.com')]
    cursor.executemany("INSERT INTO Fornecedor (ID_Fornecedor, Nome_Fornecedor, Contato_Email) VALUES (?, ?, ?)", fornecedores_data)

    # Materia_Prima
    print("  --> Inserindo dados na tabela Materia_Prima...")
    materias_primas_data = [(601, 'Liga de Carbono', 'kg'), (602, 'Motor Mini Jato', 'unidade')]
    cursor.executemany("INSERT INTO Materia_Prima (ID_Materia_Prima, Nome_Materia_Prima, Unidade_Medida) VALUES (?, ?, ?)", materias_primas_data)

    # Receita_Produto
    print("  --> Inserindo dados na tabela Receita_Produto...")
    receitas_data = [(701, 203)]
    cursor.executemany("INSERT INTO Receita_Produto (ID_Receita, ID_Produto) VALUES (?, ?)", receitas_data)

    # Item_Receita
    print("  --> Inserindo dados na tabela Item_Receita...")
    itens_receita_data = [(1, 701, 601, 5.5)]
    cursor.executemany("INSERT INTO Item_Receita (ID_Item_Receita, ID_Receita, ID_Materia_Prima, Quantidade_Necessaria) VALUES (?, ?, ?, ?)", itens_receita_data)

    # Producao_Ordem
    print("  --> Inserindo dados na tabela Producao_Ordem...")
    producao_ordens_data = [(801, 203, 100, '2024-07-01', 'Em Progresso')]
    cursor.executemany("INSERT INTO Producao_Ordem (ID_Ordem_Producao, ID_Produto, Quantidade_Produzir, Data_Inicio_Prevista, Status_Producao) VALUES (?, ?, ?, ?, ?)", producao_ordens_data)

    # Funcionario
    print("  --> Inserindo dados na tabela Funcionario...")
    funcionarios_data = [(401, 'Ana Silva', 'ana.silva@empresa.com', '2020-01-15'),
                         (402, 'Bruno Costa', 'bruno.costa@empresa.com', '2021-03-20'),
                         (403, 'Carlos Rocha', 'carlos.rocha@empresa.com', '2019-07-01')]
    cursor.executemany("INSERT INTO Funcionario (ID_Funcionario, Nome_Funcionario, Email_Funcionario, Data_Contratacao) VALUES (?, ?, ?, ?)", funcionarios_data)

    # Cargo
    print("  --> Inserindo dados na tabela Cargo...")
    cargos_data = [(1001, 'Gerente de Vendas'), (1002, 'Analista de TI'), (1003, 'Diretor de Produção')]
    cursor.executemany("INSERT INTO Cargo (ID_Cargo, Nome_Cargo) VALUES (?, ?)", cargos_data)

    # Funcionario_Cargo
    print("  --> Inserindo dados na tabela Funcionario_Cargo...")
    func_cargos_data = [(1, 401, 1001), (2, 402, 1002), (3, 403, 1003)]
    cursor.executemany("INSERT INTO Funcionario_Cargo (ID_Funcionario_Cargo, ID_Funcionario, ID_Cargo) VALUES (?, ?, ?)", func_cargos_data)

    # Departamento
    print("  --> Inserindo dados na tabela Departamento...")
    departamentos_data = [(10, 'Vendas'), (20, 'TI'), (30, 'Produção'), (40, 'Finanças')]
    cursor.executemany("INSERT INTO Departamento (ID_Departamento, Nome_Departamento) VALUES (?, ?)", departamentos_data)

    # Fatura_Venda
    print("  --> Inserindo dados na tabela Fatura_Venda...")
    faturas_data = [(901, 301, '2024-05-11', 70000.00, 'Pago'),
                    (902, 302, '2024-05-16', 120000.00, 'Pendente'),
                    (903, 303, '2024-06-02', 100000.00, 'Pago')]
    cursor.executemany("INSERT INTO Fatura_Venda (ID_Fatura_Venda, ID_Pedido, Data_Fatura, Valor_Total, Status_Pagamento) VALUES (?, ?, ?, ?, ?)", faturas_data)

    # Conta_Contabil
    print("  --> Inserindo dados na tabela Conta_Contabil...")
    contas_contabeis_data = [(10001, '1.01.01.001', 'Ativos Circulantes - Caixa'),
                             (10002, '2.01.01.001', 'Passivos Circulantes - Fornecedores'),
                             (10003, '4.01.01.001', 'Receita Bruta de Vendas'),
                             (10004, '5.01.01.001', 'Custo dos Produtos Vendidos')]
    cursor.executemany("INSERT INTO Conta_Contabil (ID_Conta_Contabil, Numero_Conta, Nome_Conta) VALUES (?, ?, ?)", contas_contabeis_data)

    # Lancamento_Contabil
    print("  --> Inserindo dados na tabela Lancamento_Contabil (inicial)...")
    lancamentos_data = [(1, 10001, '2024-05-11', 70000.00, 'Debito'),
                        (2, 10003, '2024-05-11', 70000.00, 'Credito'),
                        (3, 10001, '2024-06-02', 100000.00, 'Debito'),
                        (4, 10003, '2024-06-02', 100000.00, 'Credito'),
                        (5, 10004, '2024-05-20', 50000.00, 'Debito'),
                        (6, 10001, '2024-05-20', 50000.00, 'Credito')]
    cursor.executemany("INSERT INTO Lancamento_Contabil (ID_Lancamento, ID_Conta_Contabil, Data_Lancamento, Valor, Tipo_Lancamento) VALUES (?, ?, ?, ?, ?)", lancamentos_data)

    # Adiciona a coluna Referencia_Documento e popula via UPDATE para manter fidelidade ao modelo original
    print("  --> Verificando/adicionando Coluna 'Referencia_Documento' em Lancamento_Contabil...")
    try:
        cursor.execute("PRAGMA table_info(Lancamento_Contabil);")
        columns_info = cursor.fetchall()
        if not any(col[1] == 'Referencia_Documento' for col in columns_info):
            cursor.execute("ALTER TABLE Lancamento_Contabil ADD COLUMN Referencia_Documento TEXT;")
            print("  Coluna 'Referencia_Documento' adicionada a Lancamento_Contabil.")

        cursor.execute("UPDATE Lancamento_Contabil SET Referencia_Documento = '901' WHERE ID_Lancamento IN (1, 2);")
        cursor.execute("UPDATE Lancamento_Contabil SET Referencia_Documento = '903' WHERE ID_Lancamento IN (3, 4);")
        conn.commit()
        print("  Coluna 'Referencia_Documento' em Lancamento_Contabil populada para casos de teste.")
    except sqlite3.OperationalError as e:
        print(f"  Aviso: Coluna 'Referencia_Documento' já pode existir ou erro ao adicionar/atualizar: {e}")
        conn.rollback()


    # Centro_Custo
    print("  --> Inserindo dados na tabela Centro_Custo...")
    centros_custo_data = [(1, 'Vendas'), (2, 'Marketing'), (3, 'Produção')]
    cursor.executemany("INSERT INTO Centro_Custo (ID_Centro_Custo, Nome_Centro_Custo) VALUES (?, ?)", centros_custo_data)

    # Lancamento_Contabil_CC
    print("  --> Inserindo dados na tabela Lancamento_Contabil_CC...")
    lancamentos_cc_data = [(1, 1, 1), (2, 2, 1), (3, 3, 1)] # ID_Lancamento_CC, ID_Lancamento, ID_Centro_Custo
    cursor.executemany("INSERT INTO Lancamento_Contabil_CC (ID_Lancamento_CC, ID_Lancamento, ID_Centro_Custo) VALUES (?, ?, ?)", lancamentos_cc_data)

    # Pagamento_Fornecedor
    print("  --> Inserindo dados na tabela Pagamento_Fornecedor...")
    pagamentos_fornecedor_data = [(1, 501, 10000.00, '2024-06-01')]
    cursor.executemany("INSERT INTO Pagamento_Fornecedor (ID_Pagamento_Fornecedor, ID_Fornecedor, Valor_Pago, Data_Pagamento) VALUES (?, ?, ?, ?)", pagamentos_fornecedor_data)

    # Projeto_TI
    print("  --> Inserindo dados na tabela Projeto_TI...")
    projetos_ti_data = [(1, 'Implantação ERP', 'Sistema de gestão empresarial', 'Ativo')]
    cursor.executemany("INSERT INTO Projeto_TI (ID_Projeto_TI, Nome_Projeto, Descricao_Projeto, Status_Projeto) VALUES (?, ?, ?, ?)", projetos_ti_data)

    # Tarefa_TI
    print("  --> Inserindo dados na tabela Tarefa_TI...")
    tarefas_ti_data = [(1, 1, 'Configurar módulos financeiros', 'Em Progresso', 402)]
    cursor.executemany("INSERT INTO Tarefa_TI (ID_Tarefa_TI, ID_Projeto_TI, Nome_Tarefa, Status_Tarefa, ID_Funcionario) VALUES (?, ?, ?, ?, ?)", tarefas_ti_data)


    conn.commit()
    print("  Dados de exemplo inseridos com sucesso no SQLite.")

    print("\nVerificação rápida de contagens de linhas no SQLite:")
    for table_name in all_tables:
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        print(f"  Tabela '{table_name}': {count} linhas.")

    print("\nCriação e população do banco de dados SQLite concluída com sucesso! ✅")

except Exception as e:
    print(f"ERRO ao criar/popular o banco de dados SQLite: {e}")
    print(f"Detalhes do erro: {e}")
    if conn:
        conn.rollback()
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()

Iniciando a criação e população do banco de dados SQLite...
  Removendo tabelas SQLite existentes (se houver)...
  Tabelas SQLite limpas.
  Criando tabelas no SQLite com base no modelo lógico...
  Tabelas SQLite criadas com sucesso.
  Populando tabelas SQLite com dados de exemplo...
  --> Inserindo dados na tabela Cliente...
  --> Inserindo dados na tabela Endereco_Cliente...
  --> Inserindo dados na tabela Categoria_Produto...
  --> Inserindo dados na tabela Produto...
  --> Inserindo dados na tabela Pedido...
  --> Inserindo dados na tabela Item_Pedido...
  --> Inserindo dados na tabela Estoque...
  --> Inserindo dados na tabela Fornecedor...
  --> Inserindo dados na tabela Materia_Prima...
  --> Inserindo dados na tabela Receita_Produto...
  --> Inserindo dados na tabela Item_Receita...
  --> Inserindo dados na tabela Producao_Ordem...
  --> Inserindo dados na tabela Funcionario...
  --> Inserindo dados na tabela Cargo...
  --> Inserindo dados na tabela Funcionario_Cargo...
  --> In

In [19]:
# Célula 17: Configuração da Cadeia LangChain para Geração e Explicação de SQL
# Esta célula define as funções e a cadeia LangChain.
# O prompt foi ajustado para ser mais coercitivo, instruindo o LLM a retornar a mensagem de "NÃO FOI POSSÍVEL"
# se a conexão entre as tabelas não for explícita e direta (via FKs ou links muito óbvios).

print("Configurando a cadeia LangChain para geração e explicação de SQL (Versão Coercitiva)...")

import sqlite3
import pandas as pd
import os
import re
from unidecode import unidecode
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from numpy.linalg import norm
import numpy as np

# As variáveis 'llm' e 'embeddings' devem estar globais da Célula 4.
# graph_schema_representation (Neo4j) e run_cypher_query (Neo4j) já devem estar globais.

# --- 1. Função para Executar SQL no SQLite ---
def execute_sqlite_query(query, db_path='/content/drive/MyDrive/00_Projeto_ICMC/database.db'): # <--- AJUSTE O CAMINHO
    """
    Executa uma consulta SQL no banco de dados SQLite e retorna os resultados.
    """
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute(query)

        results = []
        for row in cursor.fetchall():
            results.append(dict(row))

        print(f"Consulta SQL executada com sucesso. Retornou {len(results)} registros. ✅")
        return results
    except sqlite3.Error as e:
        print(f"ERRO ao executar consulta SQL no SQLite: {e}")
        print(f"Query SQL que causou o erro:\n{query}")
        return []
    finally:
        if conn:
            conn.close()

print("Função 'execute_sqlite_query' definida. ✅")

# --- 2. Obter Schema do Banco SQLite para o LLM ---
sqlite_db_path = '/content/drive/MyDrive/00_Projeto_ICMC/database.db' # <--- AJUSTE O CAMINHO
sqlite_schema = ""
try:
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables_in_db = [row[0] for row in cursor.fetchall()]

    for table in tables_in_db:
        sqlite_schema += f"CREATE TABLE {table} (\n"
        cursor.execute(f"PRAGMA table_info({table});")
        columns_info = cursor.fetchall()

        col_defs = []
        for col in columns_info:
            col_name = col[1]
            col_type = col[2]
            pk = "PRIMARY KEY" if col[5] else ""
            col_defs.append(f"  {col_name} {col_type} {pk}".strip())

        sqlite_schema += ",\n".join(col_defs)

        cursor.execute(f"PRAGMA foreign_key_list({table});")
        fks_info = cursor.fetchall()
        if fks_info:
            for fk in fks_info:
                from_col = fk[3]
                to_table = fk[2]
                to_col = fk[4]
                sqlite_schema += f",\n  FOREIGN KEY ({from_col}) REFERENCES {to_table}({to_col})"

        sqlite_schema += "\n);"

    conn.close()
    print("Esquema do banco de dados SQLite extraído com sucesso. ✅")

except Exception as e:
    print(f"ERRO ao extrair esquema SQLite: {e}")
    sqlite_schema = "Esquema indisponível."


# --- 3. Definir o Prompt Template para o LLM (AGORA MAIS COERCITIVO) ---
template_sql = """
Você é um assistente especialista em bancos de dados relacionais SQLite e sistemas de informações.
Sua tarefa é:
1.  Traduzir uma pergunta em linguagem natural para uma consulta SQL (padrão ANSI SQL) que possa ser executada em um banco de dados SQLite.
2.  Após gerar a query SQL, você DEVE EXPLICAR a lógica por trás da query. A explicação deve focar nas tabelas envolvidas, COMO ELAS SÃO UNIDAS (JOINs) e quais chaves são usadas para isso, e a finalidade de quaisquer cláusulas WHERE ou funções de agregação (COUNT, SUM, etc.).

Esquema do Banco de Dados Relacional SQLite (APENAS as tabelas e colunas, sem os dados):
{sqlite_schema}

Conhecimento Semântico e Modelagem (do Grafo Neo4j, para ajudar na inferência e nos sinônimos):
- O grafo contém: Nós :Table (Tabelas), :Column (Colunas), :Concept (Conceitos), :Property (Propriedades) com seus nomes, descrições, domínios e embeddings.
- Relações :HAS_COLUMN (Tabela -> Coluna).
- Relações :HAS_FK_TO (Tabela_Origem -> Tabela_Destino), com propriedades 'fk_column' e 'pk_column' que indicam as chaves de JOIN.
- Relações :IS_A (Hierarquias) entre Conceitos.
- Metadados de Governança (em nós de coluna): ClassificacaoConfidencialidade, ProprietarioDado, RegrasQualidadeDados, FreqAtualizacao, RetencaoDadosAnos, RegraNegocioAssociada, TermoGlossarioDMBOK.
- Para filtrar por tipos específicos de produtos (como Quadriciclo, MochilaVoadora, BikeMontanha), use a PROPRIEDADE 'Nome_Produto' da tabela Produto.
- **Tabelas Contábeis Chave (para Relatórios Financeiros, IMPORTANTES): Lancamento_Contabil, Conta_Contabil, Centro_Custo, Lancamento_Contabil_CC.**

Elementos Relevantes do Grafo (encontrados por similaridade SEMÂNTICA à sua pergunta - CONTEXTO CRÍTICO):
{formatted_relevant_elements}

Instruções para a Geração SQL e Explicação:
- Baseie a query SQL APENAS nos elementos do esquema SQLite fornecido e nos elementos relevantes do grafo.
- **A saída DEVE ter o seguinte formato EXATO:**
```sql
<SUA_QUERY_SQL_AQUI>
Use SELECT para selecionar as colunas relevantes. Você pode usar SELECT t1.*, t2.* se for solicitado "todas as colunas dessas tabelas".

Use JOIN explícito para unir tabelas, SOMENTE SE HOUVER UMA CHAVE ESTRANGEIRA (FK) CLARA E DIRETA ENTRE ELAS OU UM CAMINHO DE JOIN ÓBVIO E SEMÂNTICAMENTE INQUESTIONÁVEL NO ESQUEMA SQL FORNECIDO.

Use WHERE para filtrar.

Use GROUP BY e funções de agregação (COUNT, SUM, AVG) se a pergunta sugerir.

Regra de Ouro (REFORÇADA): Se os "Elementos Relevantes do Grafo" não contiverem informações explícitas sobre as tabelas e colunas solicitadas, ou se a conexão entre tabelas para a pergunta NÃO FOR CLARA E DIRETAMENTE DEFINIDA POR UMA FK OU UM CAMINHO ÓBVIO NO ESQUEMA SQL, a ÚNICA resposta DEVE ser: "NÃO FOI POSSÍVEL GERAR A CONSULTA SQL COM BASE NO ESQUEMA DISPONÍVEL, OU A CONEXÃO ENTRE OS DADOS NÃO É DIRETA. Por favor, revise o modelo lógico ou a pergunta."

Pergunta do Usuário: {question}
"""
prompt_sql = PromptTemplate(
    template=template_sql,
    input_variables=["sqlite_schema", "formatted_relevant_elements", "question"]
)
print("Prompt Template para geração de SQL e explicação definido. ✅")


# --- 4. Construir a Cadeia LangChain para SQL ---
if 'llm' not in globals() or llm is None:
    print("ERRO: O LLM (Gemini) não foi carregado na Célula 4. Por favor, execute a Célula 4 novamente.")
    raise ValueError("LLM não disponível.")
if 'embeddings' not in globals() or embeddings is None:
    print("ERRO: O modelo de Embeddings não foi carregado na Célula 4. Por favor, execute a Célula 4 novamente.")
    raise ValueError("Modelo de Embeddings não disponível.")


# --- Aprimoramento CRÍTICO na função de busca semântica no grafo ---
def search_graph_elements_by_keyword(query_text, limit=10): # Limite ajustado para focar nos top semânticos
    """
    Busca elementos do grafo (Tabelas, Colunas, Conceitos, Propriedades) por similaridade SEMÂNTICA de embeddings.
    Calcula o embedding da pergunta e compara com embeddings dos nós no Neo4j.
    """
    if driver is None:
        return []
    if embeddings is None:
        print("ERRO: Modelo de Embeddings não carregado. Não é possível realizar busca semântica.")
        return []

    # 1. Gerar o embedding da pergunta do usuário
    query_embedding = embeddings.embed_query(query_text)
    print(f"  DEBUG: Embedding da pergunta (primeiros 5 elementos): {query_embedding[:5]}...")
    print(f"  DEBUG: Tamanho do Embedding da pergunta: {len(query_embedding)}")
    query_embedding_np = np.array(query_embedding) # Converte para numpy array para cálculo

    # 2. Trazer todos os nós relevantes com embeddings do Neo4j
    relevant_labels = ['Table', 'Column', 'Concept', 'Property']

    cypher_get_embeddings = """
    MATCH (n)
    WHERE any(label IN labels(n) WHERE label IN $relevant_labels)
      AND n.embedding IS NOT NULL
    RETURN elementId(n) AS nodeId, labels(n) AS nodeLabels, n.name AS nodeName,
           COALESCE(n.description, n.comment) AS nodeDescription, n.domain AS nodeDomain, n.embedding AS nodeEmbedding
    """
    params_get_embeddings = {"relevant_labels": relevant_labels}

    print(f"\n  DEBUG: Recuperando embeddings de {relevant_labels} do grafo para busca semântica...")
    nodes_with_embeddings = run_cypher_query(cypher_get_embeddings, params_get_embeddings)

    print(f"  DEBUG: Total de nós com embeddings recuperados: {len(nodes_with_embeddings)}")
    if not nodes_with_embeddings:
        print("  DEBUG: Nenhum nó com embedding encontrado no grafo para busca semântica.")
        return []

    # 3. Calcular similaridade de cosseno em Python e ranquear
    scored_nodes = []
    for node in nodes_with_embeddings:
        node_embedding = np.array(node['nodeEmbedding'])

        if norm(query_embedding_np) == 0 or norm(node_embedding) == 0:
            similarity = 0.0
        else:
            similarity = np.dot(query_embedding_np, node_embedding) / (norm(query_embedding_np) * norm(node_embedding))

        node['similarity'] = similarity
        scored_nodes.append(node)

    top_n_nodes = sorted(scored_nodes, key=lambda x: x['similarity'], reverse=True)[:limit]

    min_similarity_threshold = 0.5 # Ajuste este valor se os resultados estiverem muito genéricos
    filtered_top_n_nodes = [node for node in top_n_nodes if node['similarity'] >= min_similarity_threshold]

    print(f"  DEBUG: Resultados da busca semântica no grafo ({len(filtered_top_n_nodes)} registros filtrados por similaridade > {min_similarity_threshold}):")
    if filtered_top_n_nodes:
        for r in filtered_top_n_nodes:
            print(f"    - Nome: {r['nodeName']}, Labels: {r['nodeLabels']}, Domínio: {r.get('nodeDomain', 'N/A')}, Similaridade: {r['similarity']:.4f}")
    else:
        print("    Nenhum elemento relevante encontrado na busca semântica do grafo (abaixo do limiar de similaridade).")

    return filtered_top_n_nodes


# Definimos a função para formatar elementos relevantes para o prompt
def format_relevant_elements_for_prompt(elements):
    if not elements:
        return "Nenhum elemento relevante encontrado no grafo para esta pergunta."
    formatted = []
    for elem in elements:
        labels = elem.get('nodeLabels', [])
        labels_str = ":".join(labels) if isinstance(labels, list) else str(labels)
        formatted.append(
            f"- Nome: {elem.get('nodeName')}, Tipo: {labels_str}, Domínio: {elem.get('nodeDomain', 'N/A')}, Descrição: {elem.get('nodeDescription', 'N/A')}, Similaridade: {elem.get('similarity', 0.0):.4f}"
        )
    return "\n".join(formatted)


# Custom Output Parser para remover Markdown e dividir SQL da Explicação
class SQLAndExplanationParser(StrOutputParser):
    """
    Parses the output of an LLM call to extract SQL query and its explanation.
    """
    def parse(self, text: str) -> dict:
        sql_query = ""
        explanation = ""

        sql_start_tag = "```sql"
        sql_end_tag = "```"

        sql_start_index = text.find(sql_start_tag)
        sql_end_index = text.find(sql_end_tag, sql_start_index + len(sql_start_tag))

        if sql_start_index != -1 and sql_end_index != -1:
            sql_query = text[sql_start_index + len(sql_start_tag) : sql_end_index].strip()
            explanation = text[sql_end_index + len(sql_end_tag):].strip()
        else:
            if text.strip().startswith("NÃO FOI POSSÍVEL GERAR A CONSULTA SQL"):
                sql_query = text.strip()
                explanation = ""
                return {"sql": sql_query, "explanation": "Não aplicável"}

            lines = text.strip().split('\n', 1)
            sql_query = lines[0].strip()
            if len(lines) > 1:
                explanation = lines[1].strip()
            else:
                explanation = ""

        sql_query = sql_query.strip()
        if sql_query.lower().startswith("cypher query:"):
            sql_query = sql_query[len("cypher query:"):].strip()
        if sql_query.lower().startswith("sql query:"):
            sql_query = sql_query[len("sql query:"):].strip()

        return {"sql": sql_query, "explanation": explanation}


sql_generation_chain = (
    RunnablePassthrough.assign(
        relevant_graph_elements=lambda x: search_graph_elements_by_keyword(x["question"], limit=10),
        sqlite_schema=lambda x: sqlite_schema
    )
    .assign(
        formatted_relevant_elements=lambda x: format_relevant_elements_for_prompt(x["relevant_graph_elements"])
    )
    | prompt_sql
    | llm
    | SQLAndExplanationParser()
)

print("Cadeia LangChain para geração de SQL e explicação configurada. ✅")

Configurando a cadeia LangChain para geração e explicação de SQL (Versão Coercitiva)...
Função 'execute_sqlite_query' definida. ✅
Esquema do banco de dados SQLite extraído com sucesso. ✅
Prompt Template para geração de SQL e explicação definido. ✅
Cadeia LangChain para geração de SQL e explicação configurada. ✅


In [20]:
# Célula 18: Demonstração e Execução dos Casos de Uso - Caso de Uso 1 (Relatório Contábil)
# Esta célula executa o primeiro caso de uso, gerando e validando a query SQL
# para um relatório contábil complexo que envolve múltiplos JOINs.
# A cadeia LangChain (sql_generation_chain) e a função execute_sqlite_query devem estar definidas na Célula 16.

print("\n=== Teste Caso de Uso 1: Relatório Contábil (4 Joins) ===")
accounting_query = "Gere um relatório que mostre os lançamentos contábeis, incluindo a data, o valor, o tipo, o nome da conta contábil e o centro de custo associado. Inclua todas as colunas dessas tabelas."
print(f"\nPergunta: {accounting_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_accounting = sql_generation_chain.invoke({"question": accounting_query})
sql_accounting = generated_output_accounting["sql"]
explanation_accounting = generated_output_accounting["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_accounting}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_accounting}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada no banco SQLite
if sql_accounting and not sql_accounting.startswith("NÃO FOI"):
    try:
        results_accounting = execute_sqlite_query(sql_accounting)
        print(f"\nSQL Gerado e Executado. Resultados:")
        if results_accounting:
            from tabulate import tabulate
            print(tabulate(results_accounting, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada: {e}")
        print(f"Detalhes: {e}")
else:
    print(f"\nFalha na geração/execução da query SQL: {sql_accounting}")


=== Teste Caso de Uso 1: Relatório Contábil (4 Joins) ===

Pergunta: Gere um relatório que mostre os lançamentos contábeis, incluindo a data, o valor, o tipo, o nome da conta contábil e o centro de custo associado. Inclua todas as colunas dessas tabelas.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [0.05918455123901367, -0.08591415733098984, -0.1928202509880066, -0.1681068390607834, -0.3114309310913086]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: Quantidade_Disponivel, Labels: ['Column'], Domínio: Cadeia Logística, Similaridade: 0.6741
    - Nome: ID_Conta_Contabil, Labels: ['Column'], Domínio: Financas, Similaridade: 0.6691
    - Nome: ID_Centro_Custo, Labels: ['Column'], Domínio: Financas, Simila

In [21]:
# Célula 19: Demonstração e Execução dos Casos de Uso - Caso de Uso 2 (Conexão Faltante)
# Esta célula executa o segundo caso de uso, onde o LLM deve identificar e informar
# a ausência de uma conexão direta no modelo lógico para a pergunta solicitada.
# A cadeia LangChain (sql_generation_chain) e a função execute_sqlite_query devem estar definidas na Célula 16.

print("\n=== Teste Caso de Uso 2: Conexão Faltante (Contábil vs Vendas) ===")
missing_link_query = "Preciso de um relatório que conecte os lançamentos contábeis com os dados de vendas, como o total das faturas e os clientes. Me mostre a conexão."
print(f"\nPergunta: {missing_link_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_missing = sql_generation_chain.invoke({"question": missing_link_query})
sql_missing = generated_output_missing["sql"]
explanation_missing = generated_output_missing["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_missing}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_missing}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada ou informa o comportamento esperado
if sql_missing.startswith("NÃO FOI"):
    print(f"\nComportamento esperado: LLM identificou a falta de conexão direta: {sql_missing}")
else:
    print(f"\nSQL Gerado (inesperadamente ou com gambiarra): {sql_missing}")
    try:
        results_missing = execute_sqlite_query(sql_missing)
        if results_missing:
            from tabulate import tabulate
            print(tabulate(results_missing, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL para o caso de conexão faltante.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada para conexão faltante: {e}")
        print(f"Detalhes: {e}")


=== Teste Caso de Uso 2: Conexão Faltante (Contábil vs Vendas) ===

Pergunta: Preciso de um relatório que conecte os lançamentos contábeis com os dados de vendas, como o total das faturas e os clientes. Me mostre a conexão.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.05370066687464714, -0.13206645846366882, -0.19228555262088776, -0.20640817284584045, -0.17026875913143158]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: ID_Lancamento, Labels: ['Column'], Domínio: Financas, Similaridade: 0.5370
    - Nome: ID_Pedido, Labels: ['Column'], Domínio: Financas, Similaridade: 0.5273
    - Nome: ID_Lancamento_CC, Labels: ['Column'], Domínio: Financas, Similaridade: 0.5250
    - Nome: Lancamento_Contabil, La

In [22]:
# Célula 20: Demonstração e Execução dos Casos de Uso - Caso de Uso 3 (Conexão Faltante: Funcionário vs. Centro de Custo)
# Esta célula executa um caso de uso onde o LLM deve identificar e informar
# a ausência de uma conexão direta no modelo lógico para a pergunta solicitada.
# A cadeia LangChain (sql_generation_chain) e a função execute_sqlite_query devem estar definidas na Célula 16.

print("\n=== Teste Caso de Uso 3: Conexão Faltante (Funcionário vs. Centro de Custo) ===")
missing_connection_query = "Gere um relatório que conecte os funcionários da empresa com os centros de custo, mostrando o nome do funcionário e o nome do centro de custo. Me diga como eles se relacionam."
print(f"\nPergunta: {missing_connection_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_missing = sql_generation_chain.invoke({"question": missing_connection_query})
sql_missing = generated_output_missing["sql"]
explanation_missing = generated_output_missing["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_missing}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_missing}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada ou informa o comportamento esperado
if sql_missing.startswith("NÃO FOI POSSÍVEL GERAR A CONSULTA SQL"):
    print(f"\nComportamento esperado: LLM identificou a falta de conexão direta: {sql_missing}")
else:
    print(f"\nSQL Gerado (inesperadamente ou com gambiarra): {sql_missing}")
    try:
        results_missing = execute_sqlite_query(sql_missing)
        if results_missing:
            from tabulate import tabulate
            print(tabulate(results_missing, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL para o caso de conexão faltante.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada para conexão faltante: {e}")
        print(f"Detalhes: {e}")


=== Teste Caso de Uso 3: Conexão Faltante (Funcionário vs. Centro de Custo) ===

Pergunta: Gere um relatório que conecte os funcionários da empresa com os centros de custo, mostrando o nome do funcionário e o nome do centro de custo. Me diga como eles se relacionam.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [0.30064281821250916, 0.1221136823296547, -0.13311287760734558, -0.16725224256515503, -0.43915075063705444]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: ID_Centro_Custo, Labels: ['Column'], Domínio: Financas, Similaridade: 0.7034
    - Nome: Nome_Centro_Custo, Labels: ['Column'], Domínio: Financas, Similaridade: 0.6802
    - Nome: ID_Centro_Custo, Labels: ['Column'], Domínio: Financas, Similar

In [23]:
# Célula 21: Demonstração e Execução dos Casos de Uso - Caso de Uso 4 (Uso de Ontologia: Produtos Voadores)
# Esta célula demonstra como o LLM utiliza a ontologia (camada semântica)
# para identificar e consultar produtos com base em conceitos abstratos.
# A cadeia LangChain (sql_generation_chain) e a função execute_sqlite_query devem estar definidas na Célula 16.

print("\n=== Teste Caso de Uso 4: Uso de Ontologia (Produtos Voadores) ===")
ontology_query = "Liste os diferentes tipos de produtos voadores que a empresa vende, mostrando seus nomes e descrições. Me explique como você identificou esses produtos."
print(f"\nPergunta: {ontology_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
generated_output_ontology = sql_generation_chain.invoke({"question": ontology_query})
sql_ontology = generated_output_ontology["sql"]
explanation_ontology = generated_output_ontology["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_ontology}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_ontology}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada
if sql_ontology and not sql_ontology.startswith("NÃO FOI"):
    try:
        results_ontology = execute_sqlite_query(sql_ontology)
        print(f"\nSQL Gerado e Executado. Resultados:")
        if results_ontology:
            from tabulate import tabulate
            print(tabulate(results_ontology, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada: {e}")
        print(f"Detalhes: {e}")
else:
    print(f"\nFalha na geração/execução da query SQL: {sql_ontology}")


=== Teste Caso de Uso 4: Uso de Ontologia (Produtos Voadores) ===

Pergunta: Liste os diferentes tipos de produtos voadores que a empresa vende, mostrando seus nomes e descrições. Me explique como você identificou esses produtos.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.27719351649284363, 0.14288035035133362, -0.1539510041475296, -0.22593829035758972, 0.1957913190126419]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: EquipamentoVoador, Labels: ['Concept'], Domínio: None, Similaridade: 0.7272
    - Nome: ProdutoContext, Labels: ['Concept'], Domínio: None, Similaridade: 0.6939
    - Nome: MochilaVoadora, Labels: ['Concept'], Domínio: None, Similaridade: 0.6441
    - Nome: ID_Produto, Labels: ['C

In [24]:
# Célula 22: Demonstração e Execução dos Casos de Uso - Caso de Uso 5 (Governança de Dados: Mascaramento)
# Esta célula executa o caso de uso de governança de forma simplificada.
# Ela depende da busca genérica por contexto da Célula 16, e pode ter resultados não ideais para o mascaramento.
# A cadeia LangChain (sql_generation_chain) e a função execute_sqlite_query devem estar definidas na Célula 16.

print("\n=== Teste Caso de Uso 5: Governança de Dados (Mascaramento de Campo Sensível) ===")
governance_query = "Mostre o nome, e-mail e data de cadastro dos clientes, mas mascare qualquer dado sensível no resultado."
print(f"\nPergunta: {governance_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro (da Célula 16).
generated_output_governance = sql_generation_chain.invoke({"question": governance_query})
sql_governance = generated_output_governance["sql"]
explanation_governance = generated_output_governance["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_governance}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_governance}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada
if sql_governance and not sql_governance.startswith("NÃO FOI"):
    try:
        results_governance = execute_sqlite_query(sql_governance)
        print(f"\nSQL Gerado e Executado. Resultados:")
        if results_governance:
            from tabulate import tabulate
            print(tabulate(results_governance, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada: {e}")
        print(f"Detalhes: {e}")
else:
    print(f"\nFalha na geração/execução da query SQL: {sql_governance}")


=== Teste Caso de Uso 5: Governança de Dados (Mascaramento de Campo Sensível) ===

Pergunta: Mostre o nome, e-mail e data de cadastro dos clientes, mas mascare qualquer dado sensível no resultado.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.23738567531108856, 0.10102088004350662, -0.015775347128510475, -0.4507678747177124, -0.07483650743961334]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (0 registros filtrados por similaridade > 0.5):
    Nenhum elemento relevante encontrado na busca semântica do grafo (abaixo do limiar de similaridade).

--- QUERY SQL GERADA PELO LLM ---
NÃO FOI POSSÍVEL GERAR A CONSULTA SQL COM BASE NO ESQUEMA DISPONÍVEL, OU A CONEXÃO ENTRE OS DADOS NÃO É DIRETA. Por favor, revise o modelo lógico ou a pergunta.
--- FIM DA QUERY ---

-

In [25]:
# Célula 23: Demonstração e Execução dos Casos de Uso - Caso de Uso 6 (Conexões Diretas de Produto)
# Esta célula testa a capacidade do LLM de gerar uma query SQL complexa que explora
# APENAS as 5 relações diretas da tabela Produto.

print("\n=== Teste Caso de Uso 6: Conexões Diretas de Produto ===")
full_journey_query = "Para cada produto, mostre seu nome e categoria, todos os pedidos em que ele foi incluído (com a quantidade comprada), sua quantidade disponível em estoque, as receitas de produção relacionadas e as ordens de produção associadas. Inclua todas as colunas relevantes e explique as conexões."
print(f"\nPergunta: {full_journey_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_journey = sql_generation_chain.invoke({"question": full_journey_query})
sql_journey = generated_output_journey["sql"]
explanation_journey = generated_output_journey["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_journey}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_journey}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada
if sql_journey and not sql_journey.startswith("NÃO FOI"):
    try:
        results_journey = execute_sqlite_query(sql_journey)
        print(f"\nSQL Gerado e Executado. Resultados:")
        if results_journey:
            from tabulate import tabulate
            print(tabulate(results_journey, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada: {e}")
        print(f"Detalhes: {e}")
else:
    print(f"\nFalha na geração/execução da query SQL: {sql_journey}")


=== Teste Caso de Uso 6: Conexões Diretas de Produto ===

Pergunta: Para cada produto, mostre seu nome e categoria, todos os pedidos em que ele foi incluído (com a quantidade comprada), sua quantidade disponível em estoque, as receitas de produção relacionadas e as ordens de produção associadas. Inclua todas as colunas relevantes e explique as conexões.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [0.02860591746866703, -0.09414351731538773, -0.2591436803340912, -0.1772819310426712, 0.1939496397972107]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: ProdutoContext, Labels: ['Concept'], Domínio: None, Similaridade: 0.7016
    - Nome: Quantidade_Disponivel, Labels: ['Column'], Domínio: Cadeia Logística, S

In [32]:
# Célula 24: Demonstração e Execução dos Casos de Uso - Caso de Uso 7 (Meta-Conhecimento: Relações da Tabela Produto)
# Esta célula testa a capacidade do LLM de consultar o meta-conhecimento do grafo e/ou schema
# para identificar relações diretas de uma tabela específica (Produto).

print("\n=== Teste Caso de Uso 7: Meta-Conhecimento (Relações da Tabela Produto) ===")
full_journey_query = "Quantas relações diretas a tabela Produto tem em nosso modelo de dados? Me diga quais são essas relações e as tabelas envolvidas."
print(f"\nPergunta: {full_journey_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_journey = sql_generation_chain.invoke({"question": full_journey_query})
sql_journey = generated_output_journey["sql"]
explanation_journey = generated_output_journey["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_journey}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_journey}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada
if sql_journey and not sql_journey.startswith("NÃO FOI"):
    try:
        results_journey = execute_sqlite_query(sql_journey)
        print(f"\nSQL Gerado e Executado. Resultados:")
        if results_journey:
            from tabulate import tabulate
            print(tabulate(results_journey, headers="keys", tablefmt="grid"))
        else:
            print("Nenhum resultado retornado da query SQL.")
    except Exception as e:
        print(f"ERRO ao executar a query SQL gerada: {e}")
        print(f"Detalhes: {e}")
else:
    print(f"\nFalha na geração/execução da query SQL: {sql_journey}")


=== Teste Caso de Uso 7: Meta-Conhecimento (Relações da Tabela Produto) ===

Pergunta: Quantas relações diretas a tabela Produto tem em nosso modelo de dados? Me diga quais são essas relações e as tabelas envolvidas.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.3524231016635895, -0.02542283944785595, -0.0820685401558876, -0.015204411000013351, -0.43305808305740356]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: Quantidade_Disponivel, Labels: ['Column'], Domínio: Cadeia Logística, Similaridade: 0.6631
    - Nome: temRetencaoDadosAnos, Labels: ['Property'], Domínio: None, Similaridade: 0.6201
    - Nome: Estoque, Labels: ['Table'], Domínio: Cadeia Logística, Similaridade: 0.6164
    - Nome: Materia_

In [27]:
# Célula 24.1: Demonstração e Execução dos Casos de Uso - Caso de Uso 8 (Meta-Conhecimento: Relações de Tabela com Outras Tabelas)
# Esta célula testa a capacidade do LLM de responder a uma pergunta sobre as relações
# da tabela 'Produto' com outras tabelas no modelo de dados.

print("\n=== Teste Caso de Uso 8: Meta-Conhecimento (Relações da Tabela Produto com Outras Tabelas) ===")
new_meta_knowledge_query = "Me traga um relatório sobre a tabela produto mostrando todas as relações que ela tem com outras tabelas."
print(f"\nPergunta: {new_meta_knowledge_query}")

# Invoca a cadeia LangChain para gerar a resposta.
# A cadeia ainda está configurada para tentar gerar SQL.
generated_output_new_meta_knowledge = sql_generation_chain.invoke({"question": new_meta_knowledge_query})
sql_new_meta_knowledge = generated_output_new_meta_knowledge["sql"]
explanation_new_meta_knowledge = generated_output_new_meta_knowledge["explanation"]

print(f"\n--- RESPOSTA DO LLM (QUERY SQL ou INFORMAÇÃO) ---\n{sql_new_meta_knowledge}\n--- FIM DA RESPOSTA ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_new_meta_knowledge}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada se for uma query válida
if sql_new_meta_knowledge and not sql_new_meta_knowledge.startswith("NÃO FOI"):
    if sql_new_meta_knowledge.strip().lower().startswith(("select", "with")):
        print(f"\nO LLM gerou uma query SQL. Tentando executar no SQLite...")
        try:
            # Re-define run_cypher_query localmente se a Célula 25 não foi executada antes,
            # para evitar NameError caso o ambiente tenha sido redefinido parcialmente.
            # No entanto, a forma correta é garantir que a Célula 15 (ou 25) tenha sido executada.
            # execute_sqlite_query é definida na Célula 16.
            results_new_meta_knowledge = execute_sqlite_query(sql_new_meta_knowledge)
            if results_new_meta_knowledge:
                from tabulate import tabulate
                print(f"\nSQL Gerado e Executado. Resultados:")
                print(tabulate(results_new_meta_knowledge, headers="keys", tablefmt="grid"))
            else:
                print("Nenhum resultado retornado da query SQL. Verifique se a query é válida ou se há dados.")
        except Exception as e:
            print(f"ERRO ao executar a query SQL gerada: {e}")
            print(f"Detalhes: {e}")
    else:
        print("\nO LLM gerou uma resposta textual (não é uma query SQL para execução direta).")
        print("A resposta já foi exibida acima.")
else:
    print(f"\nFalha na geração da resposta do LLM ou LLM indicou que não foi possível: {sql_new_meta_knowledge}")


=== Teste Caso de Uso 8: Meta-Conhecimento (Relações da Tabela Produto com Outras Tabelas) ===

Pergunta: Me traga um relatório sobre a tabela produto mostrando todas as relações que ela tem com outras tabelas.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.25008273124694824, 0.0720396563410759, -0.2066703736782074, -0.1225271075963974, -0.21513274312019348]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: Item_Receita, Labels: ['Table'], Domínio: Producao, Similaridade: 0.6613
    - Nome: ID_Receita, Labels: ['Column'], Domínio: Producao, Similaridade: 0.6604
    - Nome: Materia_Prima, Labels: ['Table'], Domínio: Producao, Similaridade: 0.6550
    - Nome: Receita_Produto, Labels: ['Table'], Domínio: 

In [28]:
# Célula 24.2: Demonstração e Execução dos Casos de Uso - Caso de Uso 9 (Relatório Completo de Produto com Relações)
# Esta célula testa a capacidade do LLM de gerar um relatório completo de Produto,
# incluindo todas as colunas do Produto e colunas de tabelas relacionadas via JOINs.

print("\n=== Teste Caso de Uso 9: Relatório Completo de Produto com Relações ===")
complete_product_report_query = "Me traga um relatorio completo de produto mostrando todos os campos sobre ele. Faça relacao (join) com as tabelas que tiverem relacoes e me traga as colunas delas também no relatorio."
print(f"\nPergunta: {complete_product_report_query}")

# Invoca a cadeia LangChain para gerar a SQL e sua explicação
# Os prints DEBUG da busca no grafo ocorrerão aqui dentro.
generated_output_product_report = sql_generation_chain.invoke({"question": complete_product_report_query})
sql_product_report = generated_output_product_report["sql"]
explanation_product_report = generated_output_product_report["explanation"]

print(f"\n--- QUERY SQL GERADA PELO LLM ---\n{sql_product_report}\n--- FIM DA QUERY ---")
print(f"\n--- EXPLICAÇÃO DO LLM ---\n{explanation_product_report}\n--- FIM DA EXPLICAÇÃO ---")

# Tenta executar a SQL gerada
if sql_product_report and not sql_product_report.startswith("NÃO FOI"):
    if sql_product_report.strip().lower().startswith(("select", "with")):
        print(f"\nO LLM gerou uma query SQL. Tentando executar no SQLite...")
        try:
            # execute_sqlite_query é definida na Célula 16.
            results_product_report = execute_sqlite_query(sql_product_report)
            print(f"\nSQL Gerado e Executado. Resultados:")
            if results_product_report:
                from tabulate import tabulate
                print(tabulate(results_product_report, headers="keys", tablefmt="grid"))
            else:
                print("Nenhum resultado retornado da query SQL.")
        except Exception as e:
            print(f"ERRO ao executar a query SQL gerada: {e}")
            print(f"Detalhes: {e}")
    else:
        print("\nO LLM gerou uma resposta textual (não é uma query SQL para execução direta).")
        print("A resposta já foi exibida acima.")
else:
    print(f"\nFalha na geração da resposta do LLM ou LLM indicou que não foi possível: {sql_product_report}")


=== Teste Caso de Uso 9: Relatório Completo de Produto com Relações ===

Pergunta: Me traga um relatorio completo de produto mostrando todos os campos sobre ele. Faça relacao (join) com as tabelas que tiverem relacoes e me traga as colunas delas também no relatorio.
  DEBUG: Embedding da pergunta (primeiros 5 elementos): [-0.019273266196250916, -0.046650782227516174, -0.16151762008666992, -0.0018757832003757358, -0.05403288081288338]...
  DEBUG: Tamanho do Embedding da pergunta: 384

  DEBUG: Recuperando embeddings de ['Table', 'Column', 'Concept', 'Property'] do grafo para busca semântica...
  DEBUG: Total de nós com embeddings recuperados: 278
  DEBUG: Resultados da busca semântica no grafo (10 registros filtrados por similaridade > 0.5):
    - Nome: Nome_Categoria, Labels: ['Column'], Domínio: Produto, Similaridade: 0.6346
    - Nome: Quantidade_Disponivel, Labels: ['Column'], Domínio: Cadeia Logística, Similaridade: 0.6186
    - Nome: ID_Receita, Labels: ['Column'], Domínio: Produ